In [23]:

# ============================================================
# NOTEBOOK 03 
# RQ2: MULTIVARIABLE REGRESSION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns

# Project directories
ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / "data" / "processed"
TABLES_DIR = ROOT / "outputs" / "tables"
FIGURES_DIR = ROOT / "outputs" / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load validated household dataset
household = pd.read_parquet(
    DATA_DIR / "household_analysis.parquet"
)

# Survey design variables
design_vars = ["house_wgt", "psu", "stratum"]

# Validated household outcomes
outcomes = ["travel30_hh", "public_sector_hh"]

# Validate required columns
required = design_vars + outcomes + [
    "province_hh",
    "metro_hh",
    "geotype_hh",
    "income_pc_hh",
    "food_insecure_any_hh",
    "vehicle_owner_hh",
    "any_medical_aid_hh",
    "household_size_hh",
    "children_under18_hh",
    "older_adults60_hh",
    "health_transport_hh"
]

missing = [col for col in required if col not in household.columns]
assert not missing, f"Missing columns: {missing}"

# Verify survey design
assert household[design_vars].notna().all().all()
assert (household["house_wgt"] > 0).all()

# Verify binary outcome coding
for outcome in outcomes:
    assert set(household[outcome].dropna().unique()).issubset({0, 1})

print("NOTEBOOK 03: MULTIVARIABLE REGRESSION")
print("-" * 65)
print("Household dataset:", household.shape)
print("Unique PSUs:", household["psu"].nunique())
print("Unique strata:", household["stratum"].nunique())

print("\nOUTCOME DISTRIBUTIONS")
for outcome in outcomes:
    print(f"\n{outcome}")
    print(household[outcome].value_counts(dropna=False))

print("\nDATA LOADING AND VALIDATION SUCCESSFUL")


NOTEBOOK 03: MULTIVARIABLE REGRESSION
-----------------------------------------------------------------
Household dataset: (20095, 18)
Unique PSUs: 3226
Unique strata: 248

OUTCOME DISTRIBUTIONS

travel30_hh
travel30_hh
0       16887
1        3184
<NA>       24
Name: count, dtype: Int64

public_sector_hh
public_sector_hh
1       15537
0        4547
<NA>       11
Name: count, dtype: Int64

DATA LOADING AND VALIDATION SUCCESSFUL


In [24]:
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# This formats floats to show up to 2 decimal places without scientific notation
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [25]:

# ============================================================
# SURVEY-WEIGHTED HOUSEHOLD INCOME QUINTILES
# ============================================================

# Households with valid per-capita income
income_data = household.loc[
    household["income_pc_hh"].notna(),
    ["income_pc_hh", "house_wgt"]
].copy()

# Aggregate sampling weights at each distinct income value
income_distribution = (
    income_data
    .groupby("income_pc_hh", as_index=False)["house_wgt"]
    .sum()
    .sort_values("income_pc_hh")
)

# Cumulative share of weighted household population
income_distribution["cumulative_share"] = (
    income_distribution["house_wgt"].cumsum()
    / income_distribution["house_wgt"].sum()
)

# Assign quintiles using cumulative survey weights
income_distribution["income_quintile_hh"] = (
    np.searchsorted(
        [0.2, 0.4, 0.6, 0.8],
        income_distribution["cumulative_share"].to_numpy(),
        side="left"
    ) + 1
)

# Map quintiles back to the household dataset
quintile_map = income_distribution.set_index(
    "income_pc_hh"
)["income_quintile_hh"]

household["income_quintile_hh"] = (
    household["income_pc_hh"]
    .map(quintile_map)
    .astype("Int64")
)

# Verify quintile distributions
quintile_summary = (
    household
    .groupby("income_quintile_hh", dropna=False)
    .agg(
        Households=("uqnr", "size"),
        Minimum_income=("income_pc_hh", "min"),
        Maximum_income=("income_pc_hh", "max")
    )
    .reset_index()
)

print("SURVEY-WEIGHTED HOUSEHOLD INCOME QUINTILES")
print("-" * 70)

display(quintile_summary.round(2))

# Validation
assert household["income_quintile_hh"].notna().sum() == 19663
assert household["income_quintile_hh"].isna().sum() == 432

print("\nIncome quintiles reconstructed successfully.")


SURVEY-WEIGHTED HOUSEHOLD INCOME QUINTILES
----------------------------------------------------------------------


,income_quintile_hh,Households,Minimum_income,Maximum_income
0,1,4192,0.00,786.25
1,2,4432,786.67,1614.25
2,3,4205,1615.00,2999.00
3,4,3566,3000.00,6951.60
4,5,3268,6958.00,535000.00
5,<NA>,432,NaN,NaN



Income quintiles reconstructed successfully.


In [26]:

# ============================================================
# GEOGRAPHIC PREDICTOR VALIDATION
# ============================================================

print("GEOGRAPHIC PREDICTOR VALIDATION")
print("-" * 70)

# 1. Provincial distribution
print("\n1. PROVINCE")
display(
    household["province_hh"]
    .value_counts(dropna=False)
    .rename_axis("Province")
    .reset_index(name="Households")
)

# 2. Metropolitan status
print("\n2. METROPOLITAN STATUS")
display(
    household["metro_hh"]
    .value_counts(dropna=False)
    .rename_axis("Metro status")
    .reset_index(name="Households")
)

# 3. Geographic type codes
print("\n3. GEOGRAPHIC TYPE")
display(
    household["geotype_hh"]
    .value_counts(dropna=False)
    .rename_axis("Geotype code")
    .reset_index(name="Households")
)

# 4. Metro status by geographic type
print("\n4. METRO STATUS × GEOGRAPHIC TYPE")
display(
    pd.crosstab(
        household["geotype_hh"],
        household["metro_hh"],
        margins=True
    )
)

# 5. Province by metropolitan status
print("\n5. PROVINCE × METROPOLITAN STATUS")
display(
    pd.crosstab(
        household["province_hh"],
        household["metro_hh"],
        margins=True
    )
)

# 6. Identify whether geographic type determines metro status
geotype_metro_counts = (
    household.groupby("geotype_hh")["metro_hh"]
    .nunique(dropna=True)
)

print("\nDistinct metro categories within each geotype:")
print(geotype_metro_counts)

print("\nGeographic predictor validation completed.")


GEOGRAPHIC PREDICTOR VALIDATION
----------------------------------------------------------------------

1. PROVINCE


,Province,Households
0,Gauteng,5316
1,KwaZulu-Natal,3160
2,Eastern Cape,2638
3,Limpopo,2244
4,Western Cape,1807
5,Mpumalanga,1568
6,North West,1311
7,Free State,1193
8,Northern Cape,858



2. METROPOLITAN STATUS


,Metro status,Households
0,Non-metro,12124
1,Metro,7971



3. GEOGRAPHIC TYPE


,Geotype code,Households
0,1,13027
1,2,6347
2,3,721



4. METRO STATUS × GEOGRAPHIC TYPE


metro_hh,Metro,Non-metro,All
geotype_hh,,,
1,7517,5510,13027
2,369,5978,6347
3,85,636,721
All,7971,12124,20095



5. PROVINCE × METROPOLITAN STATUS


metro_hh,Metro,Non-metro,All
province_hh,,,
Eastern Cape,817,1821,2638
Free State,362,831,1193
Gauteng,4576,740,5316
KwaZulu-Natal,1065,2095,3160
Limpopo,0,2244,2244
Mpumalanga,0,1568,1568
North West,0,1311,1311
Northern Cape,0,858,858
Western Cape,1151,656,1807



Distinct metro categories within each geotype:
geotype_hh
1    2
2    2
3    2
Name: metro_hh, dtype: int64

Geographic predictor validation completed.


In [27]:

# ============================================================
# GEOGRAPHIC COMBINATIONS AND MODEL IDENTIFIABILITY
# ============================================================

# Counts for all observed geographic combinations
geo_combinations = (
    household
    .groupby(
        ["province_hh", "metro_hh", "geotype_hh"],
        observed=True
    )
    .size()
    .reset_index(name="Households")
    .sort_values(
        ["province_hh", "metro_hh", "geotype_hh"]
    )
)

print("GEOGRAPHIC COMBINATIONS")
print("-" * 70)

display(geo_combinations)

print("\nCOMBINATION SUMMARY")
print("-" * 70)

print("Observed combinations:", len(geo_combinations))
print("Minimum households:", geo_combinations["Households"].min())
print("Median households:", geo_combinations["Households"].median())

print(
    "Combinations with fewer than 30 households:",
    (geo_combinations["Households"] < 30).sum()
)

# Check the rank of the proposed additive geographic model
import patsy
import numpy.linalg as la

geo_design = patsy.dmatrix(
    "C(province_hh, Treatment(reference='Western Cape'))"
    " + C(metro_hh, Treatment(reference='Metro'))"
    " + C(geotype_hh)",
    data=household,
    return_type="dataframe"
)

matrix_rank = la.matrix_rank(geo_design.to_numpy())

print("\nADDITIVE GEOGRAPHIC MODEL MATRIX")
print("-" * 70)
print("Design matrix columns:", geo_design.shape[1])
print("Matrix rank:", matrix_rank)
print("Full column rank:", matrix_rank == geo_design.shape[1])

assert matrix_rank == geo_design.shape[1], (
    "The proposed geographic model is rank deficient."
)

print("\nGeographic model identifiability check completed.")


GEOGRAPHIC COMBINATIONS
----------------------------------------------------------------------


,province_hh,metro_hh,geotype_hh,Households
0,Eastern Cape,Metro,1,755
1,Eastern Cape,Metro,2,54
2,Eastern Cape,Metro,3,8
3,Eastern Cape,Non-metro,1,538
4,Eastern Cape,Non-metro,2,1257
5,Eastern Cape,Non-metro,3,26
6,Free State,Metro,1,292
7,Free State,Metro,2,62
8,Free State,Metro,3,8
9,Free State,Non-metro,1,701



COMBINATION SUMMARY
----------------------------------------------------------------------
Observed combinations: 39
Minimum households: 2
Median households: 193.0
Combinations with fewer than 30 households: 5

ADDITIVE GEOGRAPHIC MODEL MATRIX
----------------------------------------------------------------------
Design matrix columns: 12
Matrix rank: 12
Full column rank: True

Geographic model identifiability check completed.


In [28]:

# ============================================================
# OUTCOME-SPECIFIC COMPLETE-CASE MODEL SAMPLES
# ============================================================

# Variables required across the primary M0–M2 models
primary_predictors = [
    "province_hh",
    "metro_hh",
    "geotype_hh",
    "income_quintile_hh",
    "food_insecure_any_hh",
    "household_size_hh",
    "children_under18_hh",
    "older_adults60_hh",
    "vehicle_owner_hh",
    "any_medical_aid_hh"
]

design_variables = [
    "house_wgt",
    "psu",
    "stratum"
]

# Prepare outcome-specific common samples
model_samples = {}

sample_summary = []

for outcome in ["travel30_hh", "public_sector_hh"]:

    required = (
        design_variables
        + [outcome]
        + primary_predictors
    )

    sample = household.dropna(
        subset=required
    ).copy()

    sample = sample.loc[
        sample["house_wgt"] > 0
    ].copy()

    # Convert binary outcomes to standard integer type
    sample[outcome] = sample[outcome].astype(int)

    # Convert income quintiles to standard integer type
    sample["income_quintile_hh"] = (
        sample["income_quintile_hh"].astype(int)
    )

    model_samples[outcome] = sample

    sample_summary.append({
        "Outcome": outcome,
        "Total households": len(household),
        "Complete cases": len(sample),
        "Excluded": len(household) - len(sample),
        "Retention (%)": len(sample) / len(household) * 100,
        "PSUs": sample["psu"].nunique(),
        "Strata": sample["stratum"].nunique(),
        "Design df": (
            sample[["stratum", "psu"]]
            .drop_duplicates()
            .shape[0]
            - sample["stratum"].nunique()
        )
    })

regression_samples_summary = pd.DataFrame(
    sample_summary
)

print("PRIMARY REGRESSION SAMPLE VALIDATION")
print("-" * 75)

display(regression_samples_summary.round(2))

# Confirm no missing primary model variables
for outcome, sample in model_samples.items():

    required = (
        design_variables
        + [outcome]
        + primary_predictors
    )

    assert sample[required].notna().all().all()

    assert sample["house_wgt"].gt(0).all()

print("\nPrimary regression samples prepared successfully.")


PRIMARY REGRESSION SAMPLE VALIDATION
---------------------------------------------------------------------------


,Outcome,Total households,Complete cases,Excluded,Retention (%),PSUs,Strata,Design df
0,travel30_hh,20095,19481,614,96.94,3212,248,2964
1,public_sector_hh,20095,19494,601,97.01,3212,248,2964



Primary regression samples prepared successfully.


In [29]:

# ============================================================
# CHECK R AND SURVEY PACKAGE AVAILABILITY
# ============================================================

import shutil
import subprocess

print("R SURVEY ANALYSIS ENVIRONMENT")
print("-" * 70)

# Locate R executable
r_path = shutil.which("Rscript")

print("Rscript executable:", r_path)

if r_path is None:
    print("\nRscript was not found on the system PATH.")
    print("R may be missing or not configured in PATH.")

else:
    # Check R version
    version_result = subprocess.run(
        [r_path, "--version"],
        capture_output=True,
        text=True
    )

    print("\nR version:")
    print(
        version_result.stdout.strip()
        or version_result.stderr.strip()
    )

    # Check survey package
    check_command = (
        'if (requireNamespace("survey", quietly=TRUE)) '
        '{cat("SURVEY_AVAILABLE\\n"); '
        'cat(as.character(packageVersion("survey")))} '
        'else {cat("SURVEY_NOT_INSTALLED")}'
    )

    result = subprocess.run(
        [r_path, "-e", check_command],
        capture_output=True,
        text=True
    )

    print("\nSurvey package status:")
    print(result.stdout.strip())

    if result.returncode != 0:
        print("\nR error:")
        print(result.stderr.strip())

print("\nEnvironment check completed.")


R SURVEY ANALYSIS ENVIRONMENT
----------------------------------------------------------------------
Rscript executable: None

Rscript was not found on the system PATH.
R may be missing or not configured in PATH.

Environment check completed.


In [30]:

# ============================================================
# LOCATE R INSTALLATION ON WINDOWS
# ============================================================

import os
from pathlib import Path

print("SEARCHING FOR AN EXISTING R INSTALLATION")
print("-" * 70)

search_roots = [
    Path(os.environ.get("ProgramFiles", "C:/Program Files")) / "R",
    Path(os.environ.get("ProgramFiles(x86)", "C:/Program Files (x86)")) / "R",
    Path.home() / "AppData" / "Local" / "Programs" / "R",
    Path("C:/R")
]

found = []

for root in search_roots:
    if root.exists():
        for executable in root.glob("**/Rscript.exe"):
            found.append(executable)

if found:
    print("Rscript installations found:\n")

    for executable in found:
        print(executable)

else:
    print("No Rscript.exe found in standard installation locations.")

print("\nSearch completed.")


SEARCHING FOR AN EXISTING R INSTALLATION
----------------------------------------------------------------------
Rscript installations found:

C:\Users\tsholofelom\AppData\Local\Programs\R\R-4.5.2\bin\Rscript.exe
C:\Users\tsholofelom\AppData\Local\Programs\R\R-4.5.2\bin\x64\Rscript.exe

Search completed.


In [31]:

# ============================================================
# DIAGNOSE RSCRIPT DLL / ENVIRONMENT CONFLICTS
# ============================================================

print("RSCRIPT ENVIRONMENT DIAGNOSTICS")
print("-" * 70)

# Create a modified environment for Rscript
clean_env = os.environ.copy()

# Remove variables that may redirect R to another installation
for key in list(clean_env):
    if key.upper().startswith(("R_HOME", "R_LIBS", "R_USER")):
        clean_env.pop(key, None)

# Avoid inherited Python/Conda DLL search paths
for key in ["PYTHONHOME", "PYTHONPATH", "CONDA_PREFIX"]:
    clean_env.pop(key, None)

# Test basic R and individual dependencies
tests = {
    "Basic R": 'cat("R_OK\\n")',
    "Matrix": 'library(Matrix); cat("MATRIX_OK\\n")',
    "survival": 'library(survival); cat("SURVIVAL_OK\\n")',
    "survey": 'library(survey); cat("SURVEY_OK\\n")'
}

for environment_name, environment in [
    ("Current environment", os.environ.copy()),
    ("Modified environment", clean_env)
]:

    print(f"\n{environment_name}")
    print("=" * 70)

    for name, command in tests.items():

        try:
            result = subprocess.run(
                [str(RSCRIPT), "--vanilla", "-e", command],
                capture_output=True,
                text=True,
                timeout=45,
                env=environment
            )

            print(
                f"{name:12s} | "
                f"Return code: {result.returncode} | "
                f"Output: {result.stdout.strip() or '(empty)'}"
            )

            if result.stderr.strip():
                print("  R messages:", result.stderr.strip()[:500])

        except subprocess.TimeoutExpired:
            print(f"{name:12s} | TIMEOUT")

print("\nRscript diagnostic tests completed.")


RSCRIPT ENVIRONMENT DIAGNOSTICS
----------------------------------------------------------------------

Current environment
Basic R      | Return code: 0 | Output: R_OK
Matrix       | Return code: 0 | Output: MATRIX_OK
survival     | Return code: 0 | Output: SURVIVAL_OK
survey       | Return code: 0 | Output: SURVEY_OK
  R messages: Loading required package: grid
Loading required package: Matrix
Loading required package: survival

Attaching package: 'survey'

The following object is masked from 'package:graphics':

    dotchart

Warning message:
package 'survey' was built under R version 4.5.3

Modified environment
Basic R      | Return code: 0 | Output: R_OK
Matrix       | Return code: 0 | Output: MATRIX_OK
survival     | Return code: 0 | Output: SURVIVAL_OK
survey       | Return code: 0 | Output: SURVEY_OK
  R messages: Loading required package: grid
Loading required package: Matrix
Loading required package: survival

Attaching package: 'survey'

The following object is masked from '

In [32]:

# ============================================================
# EXPORT REGRESSION SAMPLES FOR R
# ============================================================

R_DATA_DIR = ROOT / "data" / "regression"
R_DATA_DIR.mkdir(parents=True, exist_ok=True)

export_paths = {}

print("EXPORTING REGRESSION SAMPLES FOR R")
print("-" * 70)

for outcome, sample in model_samples.items():

    export_path = R_DATA_DIR / f"{outcome}_regression.csv"

    sample.to_csv(
        export_path,
        index=False,
        na_rep=""
    )

    # Read the file back to verify successful export
    check = pd.read_csv(export_path)

    assert len(check) == len(sample)
    assert check["psu"].nunique() == 3212
    assert check["stratum"].nunique() == 248
    assert check["house_wgt"].gt(0).all()
    assert check[outcome].notna().all()

    export_paths[outcome] = export_path

    print(f"\nOutcome: {outcome}")
    print(f"File: {export_path}")
    print(f"Households: {len(check):,}")
    print(f"PSUs: {check['psu'].nunique():,}")
    print(f"Strata: {check['stratum'].nunique():,}")

print("\nBoth regression samples exported and verified.")


EXPORTING REGRESSION SAMPLES FOR R
----------------------------------------------------------------------

Outcome: travel30_hh
File: C:\Users\tsholofelom\Documents\Research Projects\ghs-2025-healthcare-inequality\data\regression\travel30_hh_regression.csv
Households: 19,481
PSUs: 3,212
Strata: 248

Outcome: public_sector_hh
File: C:\Users\tsholofelom\Documents\Research Projects\ghs-2025-healthcare-inequality\data\regression\public_sector_hh_regression.csv
Households: 19,494
PSUs: 3,212
Strata: 248

Both regression samples exported and verified.


In [33]:

# ============================================================
# INITIALISE AND VALIDATE COMPLEX SURVEY DESIGNS
# ============================================================
import textwrap

r_script = r"""
suppressPackageStartupMessages(library(survey))

options(survey.lonely.psu = "fail")

travel <- read.csv(
    "data/regression/travel30_hh_regression.csv"
)

public <- read.csv(
    "data/regression/public_sector_hh_regression.csv"
)

travel_design <- svydesign(
    ids = ~psu,
    strata = ~stratum,
    weights = ~house_wgt,
    data = travel,
    nest = TRUE
)

public_design <- svydesign(
    ids = ~psu,
    strata = ~stratum,
    weights = ~house_wgt,
    data = public,
    nest = TRUE
)

cat("GHS 2025 SURVEY DESIGN VALIDATION\n")
cat("----------------------------------------\n")

validate_design <- function(design, data, outcome) {

    cat("\nOutcome:", outcome, "\n")
    cat("Households:", nrow(data), "\n")
    cat("PSUs:", length(unique(data$psu)), "\n")
    cat("Strata:", length(unique(data$stratum)), "\n")
    cat("Design df:", degf(design), "\n")

    estimate <- svymean(
        as.formula(paste0("~", outcome)),
        design,
        na.rm = TRUE
    )

    interval <- confint(estimate)

    cat(
        "Weighted prevalence (%):",
        round(100 * coef(estimate)[1], 3), "\n"
    )

    cat(
        "95% CI (%):",
        paste(round(100 * interval[1, ], 3),
              collapse = " to "),
        "\n"
    )

    stopifnot(
        degf(design) == 2964,
        nrow(data) > 19000
    )
}

validate_design(
    travel_design, travel, "travel30_hh"
)

validate_design(
    public_design, public, "public_sector_hh"
)

cat("\nSURVEY_DESIGNS_VALIDATED\n")
"""

script_path = ROOT / "data" / "regression" / "validate_survey_design.R"
script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=120
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "SURVEY_DESIGNS_VALIDATED" in result.stdout
    print("\nSurvey design validation successful.")
else:
    print("\nSurvey design validation failed; inspect the messages above.")


R RETURN CODE: 0

R OUTPUT:
GHS 2025 SURVEY DESIGN VALIDATION
----------------------------------------

Outcome: travel30_hh 
Households: 19481 
PSUs: 3212 
Strata: 248 
Design df: 2964 
Weighted prevalence (%): 14.611 
95% CI (%): 13.848 to 15.374 

Outcome: public_sector_hh 
Households: 19494 
PSUs: 3212 
Strata: 248 
Design df: 2964 
Weighted prevalence (%): 73.748 
95% CI (%): 72.924 to 74.572 

SURVEY_DESIGNS_VALIDATED

R MESSAGES:
Warning message:
package 'survey' was built under R version 4.5.3

Survey design validation successful.


In [34]:

# ============================================================
# M0: SURVEY-WEIGHTED GEOGRAPHIC LOGISTIC REGRESSION
# ============================================================

r_script = r"""
suppressPackageStartupMessages(library(survey))

options(survey.lonely.psu = "fail")

travel <- read.csv(
    "data/regression/travel30_hh_regression.csv"
)

public <- read.csv(
    "data/regression/public_sector_hh_regression.csv"
)

prepare_data <- function(d) {
    d$province_hh <- relevel(
        factor(d$province_hh),
        ref = "Western Cape"
    )
    d$metro_hh <- relevel(
        factor(d$metro_hh),
        ref = "Metro"
    )
    d$geotype_hh <- relevel(
        factor(d$geotype_hh),
        ref = "1"
    )
    d
}

travel <- prepare_data(travel)
public <- prepare_data(public)

make_design <- function(d) {
    svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )
}

travel_design <- make_design(travel)
public_design <- make_design(public)

m0_formula <- function(outcome) {
    as.formula(
        paste(
            outcome,
            "~ province_hh + metro_hh + geotype_hh"
        )
    )
}

fit_m0 <- function(design, outcome) {

    formula <- m0_formula(outcome)

    fit <- svyglm(
        formula,
        design = design,
        family = quasibinomial()
    )

    X <- model.matrix(formula, data = design$variables)

    stopifnot(
        fit$converged,
        all(is.finite(coef(fit))),
        qr(X)$rank == ncol(X)
    )

    estimates <- coef(fit)
    standard_errors <- sqrt(diag(vcov(fit)))

    # Design-based t critical value
    df <- degf(design)
    critical <- qt(0.975, df = df)

    results <- data.frame(
        Outcome = outcome,
        Term = names(estimates),
        Coefficient = as.numeric(estimates),
        SE = as.numeric(standard_errors),
        aOR = exp(estimates),
        CI_lower = exp(estimates - critical * standard_errors),
        CI_upper = exp(estimates + critical * standard_errors),
        row.names = NULL
    )

    cat("\nOUTCOME:", outcome, "\n")
    cat("Households:", nrow(design$variables), "\n")
    cat("Design df:", df, "\n")
    cat("Converged:", fit$converged, "\n")
    cat("Model matrix rank:", qr(X)$rank, "/", ncol(X), "\n")

    print(
        results[, c("Term", "aOR", "CI_lower", "CI_upper")],
        row.names = FALSE,
        digits = 4
    )

    return(results)
}

cat("M0: SURVEY-WEIGHTED GEOGRAPHIC MODELS\n")
cat("========================================\n")

travel_results <- fit_m0(
    travel_design, "travel30_hh"
)

public_results <- fit_m0(
    public_design, "public_sector_hh"
)

all_results <- rbind(travel_results, public_results)

write.csv(
    all_results,
    "outputs/tables/m0_geographic_regression.csv",
    row.names = FALSE
)

cat("\nM0_MODELS_COMPLETED\n")
"""

script_path = ROOT / "data" / "regression" / "fit_m0.R"
script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "M0_MODELS_COMPLETED" in result.stdout
    print("\nM0 geographic models estimated successfully.")
else:
    print("\nM0 estimation failed. Inspect the messages above.")


R RETURN CODE: 0

R OUTPUT:
M0: SURVEY-WEIGHTED GEOGRAPHIC MODELS

OUTCOME: travel30_hh 
Households: 19481 
Design df: 2964 
Converged: TRUE 
Model matrix rank: 12 / 12 
                     Term    aOR CI_lower CI_upper
              (Intercept) 0.0374  0.02806  0.04986
  province_hhEastern Cape 2.6860  1.95990  3.68120
    province_hhFree State 2.7376  1.90899  3.92600
       province_hhGauteng 1.3856  1.01294  1.89546
 province_hhKwaZulu-Natal 2.0278  1.47353  2.79054
       province_hhLimpopo 1.7593  1.25606  2.46405
    province_hhMpumalanga 2.5708  1.82577  3.61989
    province_hhNorth West 3.0332  2.12635  4.32675
 province_hhNorthern Cape 2.7724  1.89305  4.06022
        metro_hhNon-metro 1.6532  1.37405  1.98902
              geotype_hh2 3.0283  2.55882  3.58390
              geotype_hh3 4.6971  3.52831  6.25302

OUTCOME: public_sector_hh 
Households: 19494 
Design df: 2964 
Converged: TRUE 
Model matrix rank: 12 / 12 
                     Term    aOR CI_lower CI_upper
       

In [35]:

# ============================================================
# M0 DESIGN-BASED JOINT TESTS AND DIAGNOSTICS
# ============================================================

r_script = r"""
suppressPackageStartupMessages(library(survey))

options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    d$province_hh <- relevel(
        factor(d$province_hh), ref = "Western Cape"
    )
    d$metro_hh <- relevel(
        factor(d$metro_hh), ref = "Metro"
    )
    d$geotype_hh <- relevel(
        factor(d$geotype_hh), ref = "1"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    formula <- as.formula(
        paste(
            outcome,
            "~ province_hh + metro_hh + geotype_hh"
        )
    )

    fit <- svyglm(
        formula,
        design = design,
        family = quasibinomial()
    )

    cat("\nOUTCOME:", outcome, "\n")
    cat("----------------------------------------\n")
    cat("Converged:", fit$converged, "\n")
    cat("Design df:", degf(design), "\n")
    cat("Model residual df:", fit$df.residual, "\n")
    cat("Maximum coefficient SE:",
        round(max(sqrt(diag(vcov(fit)))), 4), "\n")

    cat("\nJOINT DESIGN-BASED WALD TESTS\n")

    for (term in c(
        "province_hh",
        "metro_hh",
        "geotype_hh"
    )) {

        test <- regTermTest(
            fit,
            as.formula(paste0("~", term)),
            method = "Wald"
        )

        cat("\n", term, "\n", sep = "")
        print(test)
    }
}

cat("\nM0_DIAGNOSTICS_COMPLETED\n")
"""

script_path = ROOT / "data" / "regression" / "diagnose_m0.R"
script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "M0_DIAGNOSTICS_COMPLETED" in result.stdout
    print("\nM0 diagnostics completed successfully.")
else:
    print("\nM0 diagnostics failed. Review the messages above.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 
----------------------------------------
Converged: TRUE 
Design df: 2964 
Model residual df: 2953 
Maximum coefficient SE: 0.1946 

JOINT DESIGN-BASED WALD TESTS

province_hh
Wald test for province_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  10.41999  on  8  and  2953  df: p= 1.6909e-14 

metro_hh
Wald test for metro_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  28.40607  on  1  and  2953  df: p= 1.0576e-07 

geotype_hh
Wald test for geotype_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  108.0497  on  2  and  2953  df: p= < 2.22e-16 

OUTCOME: public_sector_hh 
----------------------------------------
Converged: TRUE 
Design df: 2964 
Model residual df: 2953 
Maximum coefficient SE: 0.1479 

JOINT DESIGN-BASED WALD TESTS

province_hh
Wald test for province_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())


In [36]:

# ============================================================
# VALIDATE M1 SOCIOECONOMIC PREDICTORS
# ============================================================

print("M1 SOCIOECONOMIC PREDICTOR VALIDATION")
print("-" * 70)

# Use the travel-time complete-case sample for initial checks
sample = model_samples["travel30_hh"]

categorical_vars = [
    "income_quintile_hh",
    "food_insecure_any_hh"
]

continuous_vars = [
    "household_size_hh",
    "children_under18_hh",
    "older_adults60_hh"
]

print("\n1. CATEGORICAL PREDICTORS")

for variable in categorical_vars:
    print(f"\n{variable}")
    display(
        sample[variable]
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis("Category")
        .reset_index(name="Households")
    )

print("\n2. HOUSEHOLD COMPOSITION")

display(
    sample[continuous_vars]
    .describe()
    .round(2)
)

print("\n3. CORRELATION MATRIX")

display(
    sample[continuous_vars]
    .corr()
    .round(3)
)

print("\n4. HOUSEHOLD COMPOSITION CONSISTENCY")

checks = {
    "Household size >= 1":
        (sample["household_size_hh"] >= 1).all(),

    "Children <= household size":
        (
            sample["children_under18_hh"]
            <= sample["household_size_hh"]
        ).all(),

    "Older adults <= household size":
        (
            sample["older_adults60_hh"]
            <= sample["household_size_hh"]
        ).all(),

    "Children + older adults <= household size":
        (
            sample["children_under18_hh"]
            + sample["older_adults60_hh"]
            <= sample["household_size_hh"]
        ).all()
}

for name, passed in checks.items():
    print(f"{name}: {passed}")

assert all(checks.values()), (
    "Household composition consistency check failed."
)

print("\nM1 socioeconomic predictor validation completed.")


M1 SOCIOECONOMIC PREDICTOR VALIDATION
----------------------------------------------------------------------

1. CATEGORICAL PREDICTORS

income_quintile_hh


,Category,Households
0,1,4154
1,2,4392
2,3,4158
3,4,3534
4,5,3243



food_insecure_any_hh


,Category,Households
0,0,13573
1,1,5908



2. HOUSEHOLD COMPOSITION


,household_size_hh,children_under18_hh,older_adults60_hh
count,19481.00,19481.00,19481.00
mean,3.40,0.51,0.32
std,2.35,0.50,0.47
min,1.00,0.00,0.00
25%,2.00,0.00,0.00
50%,3.00,1.00,0.00
75%,5.00,1.00,1.00
max,24.00,1.00,1.00



3. CORRELATION MATRIX


,household_size_hh,children_under18_hh,older_adults60_hh
household_size_hh,1.00,0.66,0.18
children_under18_hh,0.66,1.00,-0.01
older_adults60_hh,0.18,-0.01,1.00



4. HOUSEHOLD COMPOSITION CONSISTENCY
Household size >= 1: True
Children <= household size: True
Older adults <= household size: True
Children + older adults <= household size: True

M1 socioeconomic predictor validation completed.


In [37]:

# ============================================================
# M1: SURVEY-WEIGHTED SOCIOECONOMIC LOGISTIC REGRESSION
# ============================================================

import subprocess

r_script = r"""
suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")
all_results <- list()

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    # Set categorical predictors and reference categories
    d$province_hh <- relevel(
        factor(d$province_hh), ref = "Western Cape"
    )

    d$metro_hh <- relevel(
        factor(d$metro_hh), ref = "Metro"
    )

    d$geotype_hh <- relevel(
        factor(d$geotype_hh), ref = "1"
    )

    d$income_quintile_hh <- relevel(
        factor(d$income_quintile_hh), ref = "5"
    )

    d$food_insecure_any_hh <- relevel(
        factor(d$food_insecure_any_hh), ref = "0"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    formula <- as.formula(
        paste(
            outcome,
            "~ province_hh + metro_hh + geotype_hh",
            "+ income_quintile_hh + food_insecure_any_hh",
            "+ household_size_hh + children_under18_hh",
            "+ older_adults60_hh"
        )
    )

    fit <- svyglm(
        formula,
        design = design,
        family = quasibinomial()
    )

    X <- model.matrix(formula, data = d)

    stopifnot(
        fit$converged,
        all(is.finite(coef(fit))),
        qr(X)$rank == ncol(X)
    )

    estimates <- coef(fit)
    standard_errors <- sqrt(diag(vcov(fit)))

    # Model-specific residual degrees of freedom
    df <- fit$df.residual
    critical <- qt(0.975, df = df)

    results <- data.frame(
        Outcome = outcome,
        Model = "M1",
        Term = names(estimates),
        Coefficient = as.numeric(estimates),
        SE = as.numeric(standard_errors),
        aOR = exp(estimates),
        CI_lower = exp(estimates - critical * standard_errors),
        CI_upper = exp(estimates + critical * standard_errors),
        row.names = NULL
    )

    all_results[[outcome]] <- results

    cat("\nOUTCOME:", outcome, "\n")
    cat("----------------------------------------\n")
    cat("Households:", nrow(d), "\n")
    cat("Converged:", fit$converged, "\n")
    cat("Model matrix rank:", qr(X)$rank, "/", ncol(X), "\n")
    cat("Residual df:", df, "\n")

    print(
        results[, c("Term", "aOR", "CI_lower", "CI_upper")],
        row.names = FALSE,
        digits = 4
    )
}

combined <- do.call(rbind, all_results)

write.csv(
    combined,
    "outputs/tables/m1_socioeconomic_regression.csv",
    row.names = FALSE
)

cat("\nM1_MODELS_COMPLETED\n")
"""

script_path = ROOT / "data" / "regression" / "fit_m1.R"
script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "M1_MODELS_COMPLETED" in result.stdout
    print("\nM1 socioeconomic models estimated successfully.")
else:
    print("\nM1 estimation failed. Review the messages above.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 
----------------------------------------
Households: 19481 
Converged: TRUE 
Model matrix rank: 20 / 20 
Residual df: 2945 
                     Term     aOR CI_lower CI_upper
              (Intercept) 0.03048  0.02216  0.04193
  province_hhEastern Cape 2.32355  1.70658  3.16356
    province_hhFree State 2.47320  1.72915  3.53742
       province_hhGauteng 1.26746  0.92976  1.72781
 province_hhKwaZulu-Natal 1.82426  1.32752  2.50689
       province_hhLimpopo 1.89931  1.35711  2.65813
    province_hhMpumalanga 2.45976  1.74966  3.45805
    province_hhNorth West 2.77917  1.95681  3.94713
 province_hhNorthern Cape 2.33371  1.60876  3.38536
        metro_hhNon-metro 1.55110  1.29371  1.85969
              geotype_hh2 2.69690  2.26942  3.20490
              geotype_hh3 4.74816  3.54437  6.36080
      income_quintile_hh1 1.57754  1.29535  1.92119
      income_quintile_hh2 1.34608  1.11233  1.62896
      income_quintile_hh3 1.18422  0.98340  1.

In [ ]:

# ============================================================
# M1 DESIGN-BASED JOINT TESTS
# ============================================================

r_script = r"""
suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    d$province_hh <- relevel(
        factor(d$province_hh), ref = "Western Cape"
    )
    d$metro_hh <- relevel(
        factor(d$metro_hh), ref = "Metro"
    )
    d$geotype_hh <- relevel(
        factor(d$geotype_hh), ref = "1"
    )
    d$income_quintile_hh <- relevel(
        factor(d$income_quintile_hh), ref = "5"
    )
    d$food_insecure_any_hh <- relevel(
        factor(d$food_insecure_any_hh), ref = "0"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    formula <- as.formula(
        paste(
            outcome,
            "~ province_hh + metro_hh + geotype_hh",
            "+ income_quintile_hh + food_insecure_any_hh",
            "+ household_size_hh + children_under18_hh",
            "+ older_adults60_hh"
        )
    )

    fit <- svyglm(
        formula,
        design = design,
        family = quasibinomial()
    )

    cat("\nOUTCOME:", outcome, "\n")
    cat("----------------------------------------\n")
    cat("Residual df:", fit$df.residual, "\n")

    terms_to_test <- c(
        "province_hh",
        "metro_hh",
        "geotype_hh",
        "income_quintile_hh",
        "food_insecure_any_hh",
        "household_size_hh",
        "children_under18_hh",
        "older_adults60_hh"
    )

    for (term in terms_to_test) {

        test <- regTermTest(
            fit,
            as.formula(paste0("~", term)),
            method = "Wald"
        )

        cat("\n", term, "\n", sep = "")
        print(test)
    }
}

cat("\nM1_JOINT_TESTS_COMPLETED\n")
"""

script_path = ROOT / "data" / "regression" / "test_m1.R"
script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "M1_JOINT_TESTS_COMPLETED" in result.stdout
    print("\nM1 joint tests completed successfully.")
else:
    print("\nM1 joint tests failed. Review the messages above.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 
----------------------------------------
Residual df: 2945 

province_hh
Wald test for province_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  8.394712  on  8  and  2945  df: p= 2.4685e-11 

metro_hh
Wald test for metro_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  22.50082  on  1  and  2945  df: p= 2.2005e-06 

geotype_hh
Wald test for geotype_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  91.80273  on  2  and  2945  df: p= < 2.22e-16 

income_quintile_hh
Wald test for income_quintile_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  7.672158  on  4  and  2945  df: p= 3.799e-06 

food_insecure_any_hh
Wald test for food_insecure_any_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  61.42488  on  1  and  2945  df: p= 6.3733e-15 

household_size_hh
Wald test for household_size

In [40]:

# ============================================================
# VALIDATE M2 ENABLING-RESOURCE PREDICTORS
# ============================================================

print("M2 ENABLING-RESOURCE PREDICTOR VALIDATION")
print("=" * 70)

resource_vars = [
    "vehicle_owner_hh",
    "any_medical_aid_hh"
]

for outcome, sample in model_samples.items():

    print(f"\nOUTCOME: {outcome}")
    print("-" * 70)

    print("\n1. RESOURCE VARIABLE DISTRIBUTIONS")

    for variable in resource_vars:
        print(f"\n{variable}")

        display(
            sample[variable]
            .value_counts(dropna=False)
            .sort_index()
            .rename_axis("Category")
            .reset_index(name="Households")
        )

        assert set(sample[variable].dropna().unique()).issubset(
            {0, 1}
        ), f"Unexpected coding in {variable}"

    print("\n2. RESOURCE VARIABLES BY OUTCOME")

    for variable in resource_vars:

        print(f"\n{variable} × {outcome}")

        counts = pd.crosstab(
            sample[variable],
            sample[outcome]
        )

        display(counts)

        print(
            "Minimum observed cell count:",
            int(counts.to_numpy().min())
        )

    print("\n3. VEHICLE OWNERSHIP × MEDICAL AID")

    resource_table = pd.crosstab(
        sample["vehicle_owner_hh"],
        sample["any_medical_aid_hh"]
    )

    display(resource_table)

    print(
        "Correlation:",
        round(
            sample[resource_vars]
            .astype(float)
            .corr()
            .iloc[0, 1],
            3
        )
    )

print("\nM2 enabling-resource validation completed.")


M2 ENABLING-RESOURCE PREDICTOR VALIDATION

OUTCOME: travel30_hh
----------------------------------------------------------------------

1. RESOURCE VARIABLE DISTRIBUTIONS

vehicle_owner_hh


,Category,Households
0,0,14130
1,1,5351



any_medical_aid_hh


,Category,Households
0,0,15700
1,1,3781



2. RESOURCE VARIABLES BY OUTCOME

vehicle_owner_hh × travel30_hh


travel30_hh,0,1
vehicle_owner_hh,,
0,11474,2656
1,4885,466


... (11 outputs left out)

public_sector_hh,0,1
any_medical_aid_hh,,
0,1458,14253
1,2870,913


Minimum observed cell count: 913

3. VEHICLE OWNERSHIP × MEDICAL AID


any_medical_aid_hh,0,1
vehicle_owner_hh,,
0,13110,1027
1,2601,2756


Correlation: 0.499

M2 enabling-resource validation completed.


In [ ]:

# ============================================================
# M2: FULLY ADJUSTED SURVEY-WEIGHTED LOGISTIC REGRESSION
# ============================================================

r_script = r"""
suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")
all_results <- list()

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    # Geographic reference categories
    d$province_hh <- relevel(
        factor(d$province_hh), ref = "Western Cape"
    )
    d$metro_hh <- relevel(
        factor(d$metro_hh), ref = "Metro"
    )
    d$geotype_hh <- relevel(
        factor(d$geotype_hh), ref = "1"
    )

    # Socioeconomic reference categories
    d$income_quintile_hh <- relevel(
        factor(d$income_quintile_hh), ref = "5"
    )
    d$food_insecure_any_hh <- relevel(
        factor(d$food_insecure_any_hh), ref = "0"
    )

    # Enabling-resource reference categories
    d$vehicle_owner_hh <- relevel(
        factor(d$vehicle_owner_hh), ref = "0"
    )
    d$any_medical_aid_hh <- relevel(
        factor(d$any_medical_aid_hh), ref = "0"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    formula <- as.formula(
        paste(
            outcome,
            "~ province_hh + metro_hh + geotype_hh",
            "+ income_quintile_hh + food_insecure_any_hh",
            "+ household_size_hh + children_under18_hh",
            "+ older_adults60_hh",
            "+ vehicle_owner_hh + any_medical_aid_hh"
        )
    )

    fit <- svyglm(
        formula,
        design = design,
        family = quasibinomial()
    )

    X <- model.matrix(formula, data = d)

    stopifnot(
        fit$converged,
        all(is.finite(coef(fit))),
        qr(X)$rank == ncol(X)
    )

    estimates <- coef(fit)
    standard_errors <- sqrt(diag(vcov(fit)))

    # Use model-specific residual degrees of freedom
    df <- fit$df.residual
    critical <- qt(0.975, df = df)

    results <- data.frame(
        Outcome = outcome,
        Model = "M2",
        Term = names(estimates),
        Coefficient = as.numeric(estimates),
        SE = as.numeric(standard_errors),
        aOR = exp(estimates),
        CI_lower = exp(estimates - critical * standard_errors),
        CI_upper = exp(estimates + critical * standard_errors),
        row.names = NULL
    )

    all_results[[outcome]] <- results

    cat("\nOUTCOME:", outcome, "\n")
    cat("----------------------------------------\n")
    cat("Households:", nrow(d), "\n")
    cat("PSUs:", length(unique(d$psu)), "\n")
    cat("Strata:", length(unique(d$stratum)), "\n")
    cat("Converged:", fit$converged, "\n")
    cat("Model matrix rank:", qr(X)$rank, "/", ncol(X), "\n")
    cat("Residual df:", df, "\n")
    cat(
        "Maximum coefficient SE:",
        round(max(standard_errors), 4),
        "\n"
    )

    print(
        results[, c("Term", "aOR", "CI_lower", "CI_upper")],
        row.names = FALSE,
        digits = 4
    )
}

combined <- do.call(rbind, all_results)

write.csv(
    combined,
    "outputs/tables/m2_fully_adjusted_regression.csv",
    row.names = FALSE
)

cat("\nM2_MODELS_COMPLETED\n")
"""

script_path = ROOT / "data" / "regression" / "fit_m2.R"
script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "M2_MODELS_COMPLETED" in result.stdout
    print("\nM2 fully adjusted models estimated successfully.")
else:
    print("\nM2 estimation failed. Review the messages above.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 
----------------------------------------
Households: 19481 
PSUs: 3212 
Strata: 248 
Converged: TRUE 
Model matrix rank: 22 / 22 
Residual df: 2943 
Maximum coefficient SE: 0.1906 
                     Term     aOR CI_lower CI_upper
              (Intercept) 0.03843  0.02765  0.05341
  province_hhEastern Cape 2.26962  1.66644  3.09111
    province_hhFree State 2.42256  1.69217  3.46823
       province_hhGauteng 1.25095  0.91720  1.70616
 province_hhKwaZulu-Natal 1.78417  1.29834  2.45180
       province_hhLimpopo 1.86131  1.32898  2.60687
    province_hhMpumalanga 2.44460  1.73796  3.43855
    province_hhNorth West 2.76099  1.94134  3.92672
 province_hhNorthern Cape 2.32725  1.60154  3.38180
        metro_hhNon-metro 1.55134  1.29209  1.86262
              geotype_hh2 2.64478  2.22272  3.14698
              geotype_hh3 4.70921  3.51131  6.31577
      income_quintile_hh1 1.27713  1.02684  1.58843
      income_quintile_hh2 1.09782  0.8885

In [42]:

# ============================================================
# M2 DESIGN-ADJUSTED JOINT WALD TESTS
# ============================================================

r_script = r"""
suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    d$province_hh <- relevel(
        factor(d$province_hh), ref = "Western Cape"
    )
    d$metro_hh <- relevel(
        factor(d$metro_hh), ref = "Metro"
    )
    d$geotype_hh <- relevel(
        factor(d$geotype_hh), ref = "1"
    )
    d$income_quintile_hh <- relevel(
        factor(d$income_quintile_hh), ref = "5"
    )
    d$food_insecure_any_hh <- relevel(
        factor(d$food_insecure_any_hh), ref = "0"
    )
    d$vehicle_owner_hh <- relevel(
        factor(d$vehicle_owner_hh), ref = "0"
    )
    d$any_medical_aid_hh <- relevel(
        factor(d$any_medical_aid_hh), ref = "0"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    formula <- as.formula(
        paste(
            outcome,
            "~ province_hh + metro_hh + geotype_hh",
            "+ income_quintile_hh + food_insecure_any_hh",
            "+ household_size_hh + children_under18_hh",
            "+ older_adults60_hh",
            "+ vehicle_owner_hh + any_medical_aid_hh"
        )
    )

    fit <- svyglm(
        formula,
        design = design,
        family = quasibinomial()
    )

    cat("\nOUTCOME:", outcome, "\n")
    cat("----------------------------------------\n")
    cat("Residual df:", fit$df.residual, "\n")

    terms_to_test <- c(
        "province_hh",
        "metro_hh",
        "geotype_hh",
        "income_quintile_hh",
        "food_insecure_any_hh",
        "household_size_hh",
        "children_under18_hh",
        "older_adults60_hh",
        "vehicle_owner_hh",
        "any_medical_aid_hh"
    )

    for (term in terms_to_test) {

        test <- regTermTest(
            fit,
            as.formula(paste0("~", term)),
            method = "Wald"
        )

        cat("\n", term, "\n", sep = "")
        print(test)
    }

    cat("\nCOMBINED ENABLING-RESOURCE TEST\n")

    combined_test <- regTermTest(
        fit,
        ~vehicle_owner_hh + any_medical_aid_hh,
        method = "Wald"
    )

    print(combined_test)
}

cat("\nM2_JOINT_TESTS_COMPLETED\n")
"""

script_path = ROOT / "data" / "regression" / "test_m2.R"
script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "M2_JOINT_TESTS_COMPLETED" in result.stdout
    print("\nM2 joint tests completed successfully.")
else:
    print("\nM2 joint tests failed. Review the messages above.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 
----------------------------------------
Residual df: 2943 

province_hh
Wald test for province_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  8.228859  on  8  and  2943  df: p= 4.4633e-11 

metro_hh
Wald test for metro_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  22.17064  on  1  and  2943  df: p= 2.61e-06 

geotype_hh
Wald test for geotype_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  87.84661  on  2  and  2943  df: p= < 2.22e-16 

income_quintile_hh
Wald test for income_quintile_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  4.022021  on  4  and  2943  df: p= 0.0029532 

food_insecure_any_hh
Wald test for food_insecure_any_hh
 in svyglm(formula = formula, design = design, family = quasibinomial())
F =  54.4409  on  1  and  2943  df: p= 2.073e-13 

household_size_hh
Wald test for household_size_hh


In [44]:

# ============================================================
# NESTED MODEL COMPARISON: M0, M1 AND M2
# ============================================================

r_script = r"""
suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")
summary_rows <- list()

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    d$province_hh <- relevel(
        factor(d$province_hh), ref = "Western Cape"
    )
    d$metro_hh <- relevel(
        factor(d$metro_hh), ref = "Metro"
    )
    d$geotype_hh <- relevel(
        factor(d$geotype_hh), ref = "1"
    )
    d$income_quintile_hh <- relevel(
        factor(d$income_quintile_hh), ref = "5"
    )
    d$food_insecure_any_hh <- relevel(
        factor(d$food_insecure_any_hh), ref = "0"
    )
    d$vehicle_owner_hh <- relevel(
        factor(d$vehicle_owner_hh), ref = "0"
    )
    d$any_medical_aid_hh <- relevel(
        factor(d$any_medical_aid_hh), ref = "0"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    f0 <- as.formula(paste(
        outcome,
        "~ province_hh + metro_hh + geotype_hh"
    ))

    f1 <- update(
        f0,
        ~ . + income_quintile_hh + food_insecure_any_hh
        + household_size_hh + children_under18_hh
        + older_adults60_hh
    )

    f2 <- update(
        f1,
        ~ . + vehicle_owner_hh + any_medical_aid_hh
    )

    fits <- list(
        M0 = svyglm(f0, design, family = quasibinomial()),
        M1 = svyglm(f1, design, family = quasibinomial()),
        M2 = svyglm(f2, design, family = quasibinomial())
    )

    cat("\nOUTCOME:", outcome, "\n")
    cat("========================================\n")

    cat("\nMODEL SUMMARY\n")

    for (model_name in names(fits)) {

        fit <- fits[[model_name]]

        cat(
            model_name,
            "| Coefficients:", length(coef(fit)),
            "| Residual df:", fit$df.residual,
            "| Converged:", fit$converged,
            "\n"
        )

        summary_rows[[length(summary_rows) + 1]] <- data.frame(
            Outcome = outcome,
            Model = model_name,
            Households = nrow(d),
            Coefficients = length(coef(fit)),
            Residual_df = fit$df.residual,
            Converged = fit$converged
        )
    }

    cat("\nINCREMENTAL DESIGN-ADJUSTED WALD TESTS\n")

    cat("\nM1 additional socioeconomic block:\n")

    print(regTermTest(
        fits$M1,
        ~income_quintile_hh + food_insecure_any_hh
        + household_size_hh + children_under18_hh
        + older_adults60_hh,
        method = "Wald"
    ))

    cat("\nM2 additional enabling-resource block:\n")

    print(regTermTest(
        fits$M2,
        ~vehicle_owner_hh + any_medical_aid_hh,
        method = "Wald"
    ))
}

summary_table <- do.call(rbind, summary_rows)

write.csv(
    summary_table,
    "outputs/tables/nested_model_summary.csv",
    row.names = FALSE
)

cat("\nNESTED_MODEL_COMPARISON_COMPLETED\n")
"""

script_path = ROOT / "data" / "regression" / "compare_nested_models.R"
script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "NESTED_MODEL_COMPARISON_COMPLETED" in result.stdout
    print("\nNested-model comparison completed successfully.")
else:
    print("\nNested-model comparison failed. Review the messages above.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 

MODEL SUMMARY
M0 | Coefficients: 12 | Residual df: 2953 | Converged: TRUE 
M1 | Coefficients: 20 | Residual df: 2945 | Converged: TRUE 
M2 | Coefficients: 22 | Residual df: 2943 | Converged: TRUE 

INCREMENTAL DESIGN-ADJUSTED WALD TESTS

M1 additional socioeconomic block:
Wald test for income_quintile_hh food_insecure_any_hh household_size_hh children_under18_hh older_adults60_hh
 in svyglm(formula = f1, design = design, family = quasibinomial())
F =  17.54803  on  8  and  2945  df: p= < 2.22e-16 

M2 additional enabling-resource block:
Wald test for vehicle_owner_hh any_medical_aid_hh
 in svyglm(formula = f2, design = design, family = quasibinomial())
F =  10.90142  on  2  and  2943  df: p= 1.9188e-05 

OUTCOME: public_sector_hh 

MODEL SUMMARY
M0 | Coefficients: 12 | Residual df: 2953 | Converged: TRUE 
M1 | Coefficients: 20 | Residual df: 2945 | Converged: TRUE 
M2 | Coefficients: 22 | Residual df: 2943 | Converged: TRUE 

INCREMENT

In [45]:

# ============================================================
# M2 STANDARDISED PREDICTED PROBABILITIES BY INCOME QUINTILE
# ============================================================

import subprocess

r_script = r"""
suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")
results_list <- list()

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    d$province_hh <- relevel(
        factor(d$province_hh), ref = "Western Cape"
    )
    d$metro_hh <- relevel(
        factor(d$metro_hh), ref = "Metro"
    )
    d$geotype_hh <- relevel(
        factor(d$geotype_hh), ref = "1"
    )
    d$income_quintile_hh <- relevel(
        factor(d$income_quintile_hh), ref = "5"
    )
    d$food_insecure_any_hh <- relevel(
        factor(d$food_insecure_any_hh), ref = "0"
    )
    d$vehicle_owner_hh <- relevel(
        factor(d$vehicle_owner_hh), ref = "0"
    )
    d$any_medical_aid_hh <- relevel(
        factor(d$any_medical_aid_hh), ref = "0"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    formula <- as.formula(
        paste(
            outcome,
            "~ province_hh + metro_hh + geotype_hh",
            "+ income_quintile_hh + food_insecure_any_hh",
            "+ household_size_hh + children_under18_hh",
            "+ older_adults60_hh",
            "+ vehicle_owner_hh + any_medical_aid_hh"
        )
    )

    fit <- svyglm(
        formula,
        design = design,
        family = quasibinomial()
    )

    stopifnot(fit$converged)

    weights_vector <- weights(design, type = "sampling")

    cat("\nOUTCOME:", outcome, "\n")
    cat("----------------------------------------\n")

    for (q in 1:5) {

        scenario <- d

        scenario$income_quintile_hh <- factor(
            as.character(q),
            levels = levels(d$income_quintile_hh)
        )

        predicted <- predict(
            fit,
            newdata = scenario,
            type = "response"
        )

        standardised_probability <- weighted.mean(
            as.numeric(predicted),
            w = as.numeric(weights_vector)
        )

        stopifnot(
            is.finite(standardised_probability),
            standardised_probability >= 0,
            standardised_probability <= 1
        )

        results_list[[length(results_list) + 1]] <- data.frame(
            Outcome = outcome,
            Income_quintile = paste0("Q", q),
            Probability = standardised_probability,
            Percentage = 100 * standardised_probability
        )

        cat(
            paste0("Q", q, ": "),
            sprintf("%.3f%%", 100 * standardised_probability),
            "\n",
            sep = ""
        )
    }
}

results <- do.call(rbind, results_list)

write.csv(
    results,
    "outputs/tables/m2_standardised_income_probabilities_point.csv",
    row.names = FALSE
)

cat("\nSTANDARDISED_PROBABILITIES_COMPLETED\n")
"""

script_path = (
    ROOT / "data" / "regression" /
    "standardised_income_probabilities.R"
)

script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "STANDARDISED_PROBABILITIES_COMPLETED" in result.stdout
    print("\nStandardised probability point estimates completed.")
else:
    print("\nEstimation failed. Review the messages above.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 
----------------------------------------
Q1: 16.526%
Q2: 14.761%
Q3: 13.682%
Q4: 13.121%
Q5: 13.745%

OUTCOME: public_sector_hh 
----------------------------------------
Q1: 81.548%
Q2: 80.573%
Q3: 78.651%
Q4: 73.270%
Q5: 62.112%

STANDARDISED_PROBABILITIES_COMPLETED

R MESSAGES:
Warning message:
package 'survey' was built under R version 4.5.3

Standardised probability point estimates completed.


In [46]:

# ============================================================
# DESIGN-BASED DELTA METHOD FOR STANDARDISED PROBABILITIES
# ============================================================

import subprocess

r_script = r"""
suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")
all_rows <- list()

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    d$province_hh <- relevel(factor(d$province_hh), "Western Cape")
    d$metro_hh <- relevel(factor(d$metro_hh), "Metro")
    d$geotype_hh <- relevel(factor(d$geotype_hh), "1")
    d$income_quintile_hh <- relevel(
        factor(d$income_quintile_hh), "5"
    )
    d$food_insecure_any_hh <- relevel(
        factor(d$food_insecure_any_hh), "0"
    )
    d$vehicle_owner_hh <- relevel(
        factor(d$vehicle_owner_hh), "0"
    )
    d$any_medical_aid_hh <- relevel(
        factor(d$any_medical_aid_hh), "0"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    formula <- as.formula(paste(
        outcome,
        "~ province_hh + metro_hh + geotype_hh",
        "+ income_quintile_hh + food_insecure_any_hh",
        "+ household_size_hh + children_under18_hh",
        "+ older_adults60_hh",
        "+ vehicle_owner_hh + any_medical_aid_hh"
    ))

    fit <- svyglm(
        formula, design = design, family = quasibinomial()
    )

    beta <- coef(fit)
    Vbeta <- vcov(fit)
    w <- as.numeric(weights(design, type = "sampling"))
    wn <- w / sum(w)

    # Influence contributions from the fitted survey model.
    # These account for covariance between fitted coefficients
    # and the standardisation distribution.
    score <- residuals(fit, type = "working")
    X_fit <- model.matrix(fit)
    mu_fit <- fitted(fit)
    A <- crossprod(
        X_fit,
        X_fit * as.numeric(w * mu_fit * (1 - mu_fit))
    )

    stopifnot(qr(A)$rank == ncol(A))

    # Create a stacked vector of scenario predictions and
    # their first-order linearised observation contributions.
    scenario_predictions <- list()
    scenario_gradients <- list()

    for (q in 1:5) {

        scenario <- d
        scenario$income_quintile_hh <- factor(
            rep(as.character(q), nrow(d)),
            levels = levels(d$income_quintile_hh)
        )

        Xq <- model.matrix(
            delete.response(terms(fit)),
            data = scenario,
            contrasts.arg = fit$contrasts,
            xlev = fit$xlevels
        )

        pq <- as.numeric(plogis(Xq %*% beta))

        scenario_predictions[[q]] <- pq
        scenario_gradients[[q]] <- colSums(
            Xq * as.numeric(wn * pq * (1 - pq))
        )
    }

    # The point estimates and coefficient gradients are
    # retained here. The next cell will implement the full
    # survey-design covariance for these standardised estimates.
    point_estimates <- vapply(
        scenario_predictions,
        function(p) sum(wn * p),
        numeric(1)
    )

    cat("\nOUTCOME:", outcome, "\n")
    cat("----------------------------------------\n")
    cat("Point estimates (%):\n")

    for (q in 1:5) {
        cat(
            paste0("Q", q, ": "),
            sprintf("%.3f%%", 100 * point_estimates[q]),
            "\n",
            sep = ""
        )
    }

    cat(
        "Q1 - Q5 difference (pp):",
        sprintf("%.3f", 100 * (
            point_estimates[1] - point_estimates[5]
        )),
        "\n"
    )

    stopifnot(
        all(is.finite(point_estimates)),
        all(point_estimates >= 0 & point_estimates <= 1)
    )
}

cat("\nDELTA_METHOD_PREPARATION_COMPLETED\n")
"""

script_path = (
    ROOT / "data" / "regression" /
    "standardised_income_delta_preparation.R"
)
script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "DELTA_METHOD_PREPARATION_COMPLETED" in result.stdout
    print("\nDelta-method preparation validated.")
else:
    print("\nValidation failed. Review the messages above.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 
----------------------------------------
Point estimates (%):
Q1: 16.526%
Q2: 14.761%
Q3: 13.682%
Q4: 13.121%
Q5: 13.745%
Q1 - Q5 difference (pp): 2.782 

OUTCOME: public_sector_hh 
----------------------------------------
Point estimates (%):
Q1: 81.548%
Q2: 80.573%
Q3: 78.651%
Q4: 73.270%
Q5: 62.112%
Q1 - Q5 difference (pp): 19.436 

DELTA_METHOD_PREPARATION_COMPLETED

R MESSAGES:
Warning message:
package 'survey' was built under R version 4.5.3

Delta-method preparation validated.


In [47]:

# ============================================================
# DESIGN-BASED CONFIDENCE INTERVALS:
# STANDARDISED PROBABILITIES AND Q1–Q5 CONTRAST
# ============================================================

import subprocess

r_script = r"""
suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")
all_results <- list()

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    d$province_hh <- relevel(factor(d$province_hh), "Western Cape")
    d$metro_hh <- relevel(factor(d$metro_hh), "Metro")
    d$geotype_hh <- relevel(factor(d$geotype_hh), "1")
    d$income_quintile_hh <- relevel(
        factor(d$income_quintile_hh), "5"
    )
    d$food_insecure_any_hh <- relevel(
        factor(d$food_insecure_any_hh), "0"
    )
    d$vehicle_owner_hh <- relevel(
        factor(d$vehicle_owner_hh), "0"
    )
    d$any_medical_aid_hh <- relevel(
        factor(d$any_medical_aid_hh), "0"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    formula <- as.formula(paste(
        outcome,
        "~ province_hh + metro_hh + geotype_hh",
        "+ income_quintile_hh + food_insecure_any_hh",
        "+ household_size_hh + children_under18_hh",
        "+ older_adults60_hh",
        "+ vehicle_owner_hh + any_medical_aid_hh"
    ))

    fit <- svyglm(
        formula,
        design = design,
        family = quasibinomial()
    )

    beta <- coef(fit)
    X <- model.matrix(fit)
    w <- as.numeric(weights(design, type = "sampling"))
    wn <- w / sum(w)
    mu <- as.numeric(fitted(fit))
    y <- as.numeric(d[[outcome]])

    # Weighted logistic estimating-equation derivative
    A <- crossprod(
        X,
        X * as.numeric(w * mu * (1 - mu))
    )

    stopifnot(
        fit$converged,
        qr(A)$rank == ncol(A)
    )

    Ainv <- solve(A)

    # Observation-level coefficient influence contributions
    # for the weighted estimating equation.
    score <- X * as.numeric(w * (y - mu))
    beta_influence <- score %*% t(Ainv)

    scenario_p <- matrix(
        NA_real_,
        nrow = nrow(d),
        ncol = 5
    )

    gradient <- matrix(
        NA_real_,
        nrow = length(beta),
        ncol = 5
    )

    for (q in 1:5) {

        scenario <- d

        scenario$income_quintile_hh <- factor(
            rep(as.character(q), nrow(d)),
            levels = levels(d$income_quintile_hh)
        )

        Xq <- model.matrix(
            delete.response(terms(fit)),
            data = scenario,
            contrasts.arg = fit$contrasts,
            xlev = fit$xlevels
        )

        pq <- as.numeric(plogis(Xq %*% beta))

        scenario_p[, q] <- pq

        gradient[, q] <- colSums(
            Xq * as.numeric(wn * pq * (1 - pq))
        )
    }

    probabilities <- colSums(
        scenario_p * wn
    )

    # Full influence contributions:
    # (a) standardisation-distribution variation
    # (b) fitted-regression-coefficient variation
    influence <- matrix(
        NA_real_,
        nrow = nrow(d),
        ncol = 5
    )

    for (q in 1:5) {

        distribution_part <- (
            w * (scenario_p[, q] - probabilities[q])
        ) / sum(w)

        coefficient_part <- as.numeric(
            beta_influence %*% gradient[, q]
        )

        influence[, q] <- (
            distribution_part + coefficient_part
        )
    }

    # Convert the linearised contributions to survey
    # totals and use the original PSU/stratum design.
    # The contributions already contain sampling weights.
    linearised_design <- design
    linearised_design$variables <- cbind(
        linearised_design$variables,
        as.data.frame(influence)
    )

    names(linearised_design$variables)[
        (ncol(linearised_design$variables) - 4):
        ncol(linearised_design$variables)
    ] <- paste0("lin_q", 1:5)

    covariance <- vcov(
        svytotal(
            ~lin_q1 + lin_q2 + lin_q3 + lin_q4 + lin_q5,
            design = linearised_design,
            na.rm = FALSE
        )
    )

    # svytotal applies design weights. Because influence
    # contributions above already include weights, use
    # unweighted totals through an otherwise identical
    # cluster/stratum design.
    unweighted_design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~1,
        data = cbind(
            d[, c("psu", "stratum")],
            as.data.frame(
                setNames(
                    as.data.frame(influence),
                    paste0("lin_q", 1:5)
                )
            )
        ),
        nest = TRUE
    )

    covariance <- vcov(
        svytotal(
            ~lin_q1 + lin_q2 + lin_q3 + lin_q4 + lin_q5,
            design = unweighted_design
        )
    )

    se <- sqrt(diag(covariance))

    # Survey design degrees of freedom
    df <- degf(design)
    critical <- qt(0.975, df = df)

    ci_lower <- probabilities - critical * se
    ci_upper <- probabilities + critical * se

    contrast <- c(1, 0, 0, 0, -1)

    difference <- sum(contrast * probabilities)
    difference_se <- sqrt(
        as.numeric(
            t(contrast) %*% covariance %*% contrast
        )
    )

    difference_ci <- difference +
        c(-1, 1) * critical * difference_se

    result_table <- data.frame(
        Outcome = outcome,
        Quintile = paste0("Q", 1:5),
        Probability = probabilities,
        SE = se,
        CI_lower = ci_lower,
        CI_upper = ci_upper
    )

    all_results[[outcome]] <- result_table

    cat("\nOUTCOME:", outcome, "\n")
    cat("----------------------------------------\n")
    cat("Design df:", df, "\n")

    print(
        data.frame(
            Quintile = paste0("Q", 1:5),
            Percent = round(100 * probabilities, 3),
            CI_lower = round(100 * ci_lower, 3),
            CI_upper = round(100 * ci_upper, 3)
        ),
        row.names = FALSE
    )

    cat("\nQ1 - Q5 DIFFERENCE\n")
    cat(
        "Difference (pp):",
        round(100 * difference, 3), "\n"
    )
    cat(
        "SE (pp):",
        round(100 * difference_se, 3), "\n"
    )
    cat(
        "95% CI (pp):",
        round(100 * difference_ci[1], 3),
        "to",
        round(100 * difference_ci[2], 3),
        "\n"
    )

    stopifnot(
        all(is.finite(se)),
        all(se >= 0),
        is.finite(difference_se)
    )
}

combined <- do.call(rbind, all_results)

write.csv(
    combined,
    "outputs/tables/m2_standardised_income_probabilities_ci.csv",
    row.names = FALSE
)

cat("\nSTANDARDISED_PROBABILITY_CI_COMPLETED\n")
"""

script_path = (
    ROOT / "data" / "regression" /
    "standardised_income_confidence_intervals.R"
)

script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "STANDARDISED_PROBABILITY_CI_COMPLETED" in result.stdout
    print("\nDesign-adjusted confidence intervals completed.")
else:
    print("\nConfidence interval estimation failed.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 
----------------------------------------
Design df: 2964 
 Quintile Percent CI_lower CI_upper
       Q1  16.526   15.213   17.840
       Q2  14.761   13.509   16.014
       Q3  13.682   12.416   14.947
       Q4  13.121   11.700   14.543
       Q5  13.745   11.849   15.640

Q1 - Q5 DIFFERENCE
Difference (pp): 2.782 
SE (pp): 1.237 
95% CI (pp): 0.355 to 5.208 

OUTCOME: public_sector_hh 
----------------------------------------
Design df: 2964 
 Quintile Percent CI_lower CI_upper
       Q1  81.548   79.823   83.274
       Q2  80.573   79.010   82.136
       Q3  78.651   77.423   79.880
       Q4  73.270   72.010   74.530
       Q5  62.112   60.262   63.963

Q1 - Q5 DIFFERENCE
Difference (pp): 19.436 
SE (pp): 1.341 
95% CI (pp): 16.807 to 22.065 

STANDARDISED_PROBABILITY_CI_COMPLETED

R MESSAGES:
Warning message:
package 'survey' was built under R version 4.5.3

Design-adjusted confidence intervals completed.


In [48]:

# ============================================================
# VALIDATE CUSTOM SURVEY TAYLOR LINEARISATION
# AGAINST svyglm() COEFFICIENT COVARIANCE
# ============================================================

import subprocess

r_script = r"""
suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    d$province_hh <- relevel(
        factor(d$province_hh), "Western Cape"
    )
    d$metro_hh <- relevel(
        factor(d$metro_hh), "Metro"
    )
    d$geotype_hh <- relevel(
        factor(d$geotype_hh), "1"
    )
    d$income_quintile_hh <- relevel(
        factor(d$income_quintile_hh), "5"
    )
    d$food_insecure_any_hh <- relevel(
        factor(d$food_insecure_any_hh), "0"
    )
    d$vehicle_owner_hh <- relevel(
        factor(d$vehicle_owner_hh), "0"
    )
    d$any_medical_aid_hh <- relevel(
        factor(d$any_medical_aid_hh), "0"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    formula <- as.formula(paste(
        outcome,
        "~ province_hh + metro_hh + geotype_hh",
        "+ income_quintile_hh + food_insecure_any_hh",
        "+ household_size_hh + children_under18_hh",
        "+ older_adults60_hh",
        "+ vehicle_owner_hh + any_medical_aid_hh"
    ))

    fit <- svyglm(
        formula,
        design = design,
        family = quasibinomial()
    )

    X <- model.matrix(fit)
    w <- as.numeric(weights(design, type = "sampling"))
    mu <- as.numeric(fitted(fit))
    y <- as.numeric(d[[outcome]])

    A <- crossprod(
        X,
        X * as.numeric(w * mu * (1 - mu))
    )

    stopifnot(qr(A)$rank == ncol(A))

    # First-order influence contributions for each
    # fitted logistic-regression coefficient.
    score <- X * as.numeric(w * (y - mu))

    coefficient_influence <- score %*% t(solve(A))

    colnames(coefficient_influence) <- paste0(
        "inf_", seq_len(ncol(coefficient_influence))
    )

    # Influence values already incorporate survey weights.
    # Use an unweighted design to estimate their total covariance.
    influence_data <- data.frame(
        psu = d$psu,
        stratum = d$stratum,
        coefficient_influence,
        check.names = FALSE
    )

    influence_design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~1,
        data = influence_data,
        nest = TRUE
    )

    influence_formula <- reformulate(
        colnames(coefficient_influence)
    )

    V_custom <- vcov(
        svytotal(influence_formula, influence_design)
    )

    V_survey <- vcov(fit)

    stopifnot(
        all(dim(V_custom) == dim(V_survey)),
        all(is.finite(V_custom)),
        all(is.finite(V_survey))
    )

    se_custom <- sqrt(diag(V_custom))
    se_survey <- sqrt(diag(V_survey))

    se_ratio <- se_custom / se_survey

    relative_matrix_difference <- max(
        abs(V_custom - V_survey)
    ) / max(abs(V_survey))

    eigenvalues <- eigen(
        (V_custom + t(V_custom)) / 2,
        symmetric = TRUE,
        only.values = TRUE
    )$values

    cat("\nOUTCOME:", outcome, "\n")
    cat("----------------------------------------\n")

    cat(
        "Coefficient count:",
        length(coef(fit)), "\n"
    )

    cat(
        "Survey design df:",
        degf(design), "\n"
    )

    cat(
        "Custom / survey SE ratio range:",
        sprintf("%.6f", min(se_ratio)),
        "to",
        sprintf("%.6f", max(se_ratio)),
        "\n"
    )

    cat(
        "Maximum absolute SE difference:",
        sprintf("%.8f", max(abs(
            se_custom - se_survey
        ))),
        "\n"
    )

    cat(
        "Relative covariance matrix difference:",
        sprintf("%.8f", relative_matrix_difference),
        "\n"
    )

    cat(
        "Minimum covariance eigenvalue:",
        sprintf("%.10f", min(eigenvalues)),
        "\n"
    )

    cat(
        "All coefficient SEs positive:",
        all(se_custom > 0),
        "\n"
    )

    if (relative_matrix_difference < 0.01) {
        cat("VALIDATION STATUS: CLOSE AGREEMENT\n")
    } else {
        cat("VALIDATION STATUS: REQUIRES INVESTIGATION\n")
    }
}

cat("\nINFLUENCE_COVARIANCE_VALIDATION_COMPLETED\n")
"""

script_path = (
    ROOT / "data" / "regression" /
    "validate_influence_covariance.R"
)

script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert (
        "INFLUENCE_COVARIANCE_VALIDATION_COMPLETED"
        in result.stdout
    )
    print("\nCovariance validation diagnostics completed.")
else:
    print("\nCovariance validation failed.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 
----------------------------------------
Coefficient count: 22 
Survey design df: 2964 
Custom / survey SE ratio range: 0.999993 to 1.000002 
Maximum absolute SE difference: 0.00000116 
Relative covariance matrix difference: 0.00001068 
Minimum covariance eigenvalue: 0.0000379295 
All coefficient SEs positive: TRUE 
VALIDATION STATUS: CLOSE AGREEMENT

OUTCOME: public_sector_hh 
----------------------------------------
Coefficient count: 22 
Survey design df: 2964 
Custom / survey SE ratio range: 0.999999 to 1.000002 
Maximum absolute SE difference: 0.00000015 
Relative covariance matrix difference: 0.00000150 
Minimum covariance eigenvalue: 0.0000649355 
All coefficient SEs positive: TRUE 
VALIDATION STATUS: CLOSE AGREEMENT

INFLUENCE_COVARIANCE_VALIDATION_COMPLETED

R MESSAGES:
Warning message:
package 'survey' was built under R version 4.5.3

Covariance validation diagnostics completed.


In [49]:

# ============================================================
# M2 AVERAGE MARGINAL EFFECTS — POINT ESTIMATES
# ============================================================

import subprocess

r_script = r"""
suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")

binary_predictors <- c(
    "metro_hh",
    "food_insecure_any_hh",
    "vehicle_owner_hh",
    "any_medical_aid_hh"
)

results_list <- list()

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    d$province_hh <- relevel(
        factor(d$province_hh), "Western Cape"
    )
    d$metro_hh <- relevel(
        factor(d$metro_hh), "Metro"
    )
    d$geotype_hh <- relevel(
        factor(d$geotype_hh), "1"
    )
    d$income_quintile_hh <- relevel(
        factor(d$income_quintile_hh), "5"
    )
    d$food_insecure_any_hh <- relevel(
        factor(d$food_insecure_any_hh), "0"
    )
    d$vehicle_owner_hh <- relevel(
        factor(d$vehicle_owner_hh), "0"
    )
    d$any_medical_aid_hh <- relevel(
        factor(d$any_medical_aid_hh), "0"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    formula <- as.formula(paste(
        outcome,
        "~ province_hh + metro_hh + geotype_hh",
        "+ income_quintile_hh + food_insecure_any_hh",
        "+ household_size_hh + children_under18_hh",
        "+ older_adults60_hh",
        "+ vehicle_owner_hh + any_medical_aid_hh"
    ))

    fit <- svyglm(
        formula,
        design = design,
        family = quasibinomial()
    )

    stopifnot(fit$converged)

    w <- as.numeric(weights(design, type = "sampling"))

    cat("\nOUTCOME:", outcome, "\n")
    cat("----------------------------------------\n")

    for (variable in binary_predictors) {

        scenario_0 <- d
        scenario_1 <- d

        if (variable == "metro_hh") {

            scenario_0[[variable]] <- factor(
                rep("Metro", nrow(d)),
                levels = levels(d[[variable]])
            )

            scenario_1[[variable]] <- factor(
                rep("Non-metro", nrow(d)),
                levels = levels(d[[variable]])
            )

            contrast_label <- "Non-metro vs Metro"

        } else {

            scenario_0[[variable]] <- factor(
                rep("0", nrow(d)),
                levels = levels(d[[variable]])
            )

            scenario_1[[variable]] <- factor(
                rep("1", nrow(d)),
                levels = levels(d[[variable]])
            )

            contrast_label <- paste0(variable, ": 1 vs 0")
        }

        p0 <- as.numeric(predict(
            fit,
            newdata = scenario_0,
            type = "response"
        ))

        p1 <- as.numeric(predict(
            fit,
            newdata = scenario_1,
            type = "response"
        ))

        ame <- weighted.mean(p1 - p0, w = w)

        stopifnot(
            is.finite(ame),
            ame >= -1,
            ame <= 1
        )

        results_list[[length(results_list) + 1]] <- data.frame(
            Outcome = outcome,
            Predictor = variable,
            Contrast = contrast_label,
            AME = ame,
            AME_percentage_points = 100 * ame
        )

        cat(
            contrast_label,
            ": ",
            sprintf("%+.3f pp", 100 * ame),
            "\n",
            sep = ""
        )
    }
}

results <- do.call(rbind, results_list)

write.csv(
    results,
    "outputs/tables/m2_average_marginal_effects_point.csv",
    row.names = FALSE
)

cat("\nM2_AME_POINT_ESTIMATES_COMPLETED\n")
"""

script_path = (
    ROOT / "data" / "regression" /
    "m2_average_marginal_effects_point.R"
)

script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "M2_AME_POINT_ESTIMATES_COMPLETED" in result.stdout
    print("\nM2 AME point estimates completed successfully.")
else:
    print("\nM2 AME estimation failed. Review the messages above.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 
----------------------------------------
Non-metro vs Metro: +4.660 pp
food_insecure_any_hh: 1 vs 0: +5.356 pp
vehicle_owner_hh: 1 vs 0: -3.519 pp
any_medical_aid_hh: 1 vs 0: -0.234 pp

OUTCOME: public_sector_hh 
----------------------------------------
Non-metro vs Metro: +3.113 pp
food_insecure_any_hh: 1 vs 0: +4.295 pp
vehicle_owner_hh: 1 vs 0: -16.074 pp
any_medical_aid_hh: 1 vs 0: -35.668 pp

M2_AME_POINT_ESTIMATES_COMPLETED

R MESSAGES:
Warning message:
package 'survey' was built under R version 4.5.3

M2 AME point estimates completed successfully.


In [50]:

# ============================================================
# SURVEY-DESIGN-ADJUSTED AME CONFIDENCE INTERVALS
# ============================================================

import subprocess

r_script = r"""
suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")

predictors <- c(
    "metro_hh",
    "food_insecure_any_hh",
    "vehicle_owner_hh",
    "any_medical_aid_hh"
)

all_results <- list()

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    d$province_hh <- relevel(factor(d$province_hh), "Western Cape")
    d$metro_hh <- relevel(factor(d$metro_hh), "Metro")
    d$geotype_hh <- relevel(factor(d$geotype_hh), "1")
    d$income_quintile_hh <- relevel(
        factor(d$income_quintile_hh), "5"
    )
    d$food_insecure_any_hh <- relevel(
        factor(d$food_insecure_any_hh), "0"
    )
    d$vehicle_owner_hh <- relevel(
        factor(d$vehicle_owner_hh), "0"
    )
    d$any_medical_aid_hh <- relevel(
        factor(d$any_medical_aid_hh), "0"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    formula <- as.formula(paste(
        outcome,
        "~ province_hh + metro_hh + geotype_hh",
        "+ income_quintile_hh + food_insecure_any_hh",
        "+ household_size_hh + children_under18_hh",
        "+ older_adults60_hh",
        "+ vehicle_owner_hh + any_medical_aid_hh"
    ))

    fit <- svyglm(
        formula,
        design = design,
        family = quasibinomial()
    )

    stopifnot(fit$converged)

    beta <- coef(fit)
    X <- model.matrix(fit)

    w <- as.numeric(weights(design, type = "sampling"))
    wn <- w / sum(w)

    mu <- as.numeric(fitted(fit))
    y <- as.numeric(d[[outcome]])

    A <- crossprod(
        X,
        X * as.numeric(w * mu * (1 - mu))
    )

    stopifnot(qr(A)$rank == ncol(A))

    score <- X * as.numeric(w * (y - mu))
    beta_influence <- score %*% t(solve(A))

    n <- nrow(d)
    k <- length(predictors)

    individual_differences <- matrix(
        NA_real_, nrow = n, ncol = k
    )

    gradients <- matrix(
        NA_real_, nrow = length(beta), ncol = k
    )

    labels <- character(k)

    for (j in seq_along(predictors)) {

        variable <- predictors[j]

        scenario0 <- d
        scenario1 <- d

        if (variable == "metro_hh") {

            value0 <- "Metro"
            value1 <- "Non-metro"
            labels[j] <- "Non-metro vs Metro"

        } else {

            value0 <- "0"
            value1 <- "1"
            labels[j] <- paste0(variable, ": 1 vs 0")
        }

        scenario0[[variable]] <- factor(
            rep(value0, n),
            levels = levels(d[[variable]])
        )

        scenario1[[variable]] <- factor(
            rep(value1, n),
            levels = levels(d[[variable]])
        )

        X0 <- model.matrix(
            delete.response(terms(fit)),
            data = scenario0,
            contrasts.arg = fit$contrasts,
            xlev = fit$xlevels
        )

        X1 <- model.matrix(
            delete.response(terms(fit)),
            data = scenario1,
            contrasts.arg = fit$contrasts,
            xlev = fit$xlevels
        )

        p0 <- as.numeric(plogis(X0 %*% beta))
        p1 <- as.numeric(plogis(X1 %*% beta))

        individual_differences[, j] <- p1 - p0

        gradient1 <- colSums(
            X1 * as.numeric(wn * p1 * (1 - p1))
        )

        gradient0 <- colSums(
            X0 * as.numeric(wn * p0 * (1 - p0))
        )

        gradients[, j] <- gradient1 - gradient0
    }

    ame <- colSums(
        individual_differences * wn
    )

    influence <- matrix(
        NA_real_, nrow = n, ncol = k
    )

    for (j in seq_len(k)) {

        distribution_part <- (
            w * (individual_differences[, j] - ame[j])
        ) / sum(w)

        coefficient_part <- as.numeric(
            beta_influence %*% gradients[, j]
        )

        influence[, j] <- (
            distribution_part + coefficient_part
        )
    }

    colnames(influence) <- paste0("lin_ame", seq_len(k))

    # Influence contributions already contain survey weights.
    influence_data <- data.frame(
        psu = d$psu,
        stratum = d$stratum,
        influence,
        check.names = FALSE
    )

    influence_design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~1,
        data = influence_data,
        nest = TRUE
    )

    covariance <- vcov(
        svytotal(
            reformulate(colnames(influence)),
            design = influence_design
        )
    )

    se <- sqrt(diag(covariance))

    df <- degf(design)
    critical <- qt(0.975, df = df)

    lower <- ame - critical * se
    upper <- ame + critical * se

    t_statistics <- ame / se
    p_values <- 2 * pt(
        -abs(t_statistics),
        df = df
    )

    stopifnot(
        all(is.finite(ame)),
        all(is.finite(se)),
        all(se > 0),
        all(is.finite(p_values)),
        all(lower <= ame),
        all(upper >= ame)
    )

    results <- data.frame(
        Outcome = outcome,
        Predictor = predictors,
        Contrast = labels,
        AME = ame,
        SE = se,
        CI_lower = lower,
        CI_upper = upper,
        P_value = p_values,
        AME_pp = 100 * ame,
        CI_lower_pp = 100 * lower,
        CI_upper_pp = 100 * upper
    )

    all_results[[outcome]] <- results

    cat("\nOUTCOME:", outcome, "\n")
    cat("----------------------------------------\n")
    cat("Households:", n, "\n")
    cat("Design df:", df, "\n")

    print(
        data.frame(
            Contrast = labels,
            AME_pp = round(100 * ame, 3),
            CI_lower_pp = round(100 * lower, 3),
            CI_upper_pp = round(100 * upper, 3),
            P_value = format.pval(
                p_values, digits = 3, eps = 0.0001
            )
        ),
        row.names = FALSE
    )
}

combined <- do.call(rbind, all_results)

write.csv(
    combined,
    "outputs/tables/m2_average_marginal_effects_ci.csv",
    row.names = FALSE
)

cat("\nM2_AME_CONFIDENCE_INTERVALS_COMPLETED\n")
"""

script_path = (
    ROOT / "data" / "regression" /
    "m2_average_marginal_effects_ci.R"
)

script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "M2_AME_CONFIDENCE_INTERVALS_COMPLETED" in result.stdout
    print("\nM2 AME confidence intervals completed successfully.")
else:
    print("\nM2 AME confidence interval estimation failed.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 
----------------------------------------
Households: 19481 
Design df: 2964 
                     Contrast AME_pp CI_lower_pp CI_upper_pp P_value
           Non-metro vs Metro  4.660       2.799       6.521  <1e-04
 food_insecure_any_hh: 1 vs 0  5.356       3.870       6.842  <1e-04
     vehicle_owner_hh: 1 vs 0 -3.519      -4.993      -2.045  <1e-04
   any_medical_aid_hh: 1 vs 0 -0.234      -2.067       1.600   0.803

OUTCOME: public_sector_hh 
----------------------------------------
Households: 19494 
Design df: 2964 
                     Contrast  AME_pp CI_lower_pp CI_upper_pp P_value
           Non-metro vs Metro   3.113       1.749       4.476  <1e-04
 food_insecure_any_hh: 1 vs 0   4.295       3.040       5.551  <1e-04
     vehicle_owner_hh: 1 vs 0 -16.074     -17.716     -14.431  <1e-04
   any_medical_aid_hh: 1 vs 0 -35.668     -37.968     -33.368  <1e-04

M2_AME_CONFIDENCE_INTERVALS_COMPLETED

R MESSAGES:
Warning message:
pack

In [52]:

# ============================================================
# CONSOLIDATE REGRESSION AND MARGINAL-EFFECTS TABLES
# ============================================================

# ------------------------------------------------------------
# 1. Load and validate regression outputs
# ------------------------------------------------------------

model_files = {
    "M0": TABLES_DIR / "m0_geographic_regression.csv",
    "M1": TABLES_DIR / "m1_socioeconomic_regression.csv",
    "M2": TABLES_DIR / "m2_fully_adjusted_regression.csv",
}

models = {}

for model_name, path in model_files.items():

    assert path.exists(), f"Missing file: {path}"

    df = pd.read_csv(path)

    required = {
        "Outcome", "Term", "aOR", "CI_lower", "CI_upper"
    }

    assert required.issubset(df.columns), (
        f"{model_name}: missing {required - set(df.columns)}"
    )

    assert set(df["Outcome"]) == {
        "travel30_hh", "public_sector_hh"
    }

    assert (
        (df["aOR"] > 0) &
        (df["CI_lower"] > 0) &
        (df["CI_upper"] > 0) &
        (df["CI_lower"] <= df["aOR"]) &
        (df["aOR"] <= df["CI_upper"])
    ).all()

    models[model_name] = df.copy()

# ------------------------------------------------------------
# 2. Consistent labels
# ------------------------------------------------------------

term_labels = {
    "(Intercept)": "Intercept",
    "province_hhEastern Cape": "Eastern Cape",
    "province_hhFree State": "Free State",
    "province_hhGauteng": "Gauteng",
    "province_hhKwaZulu-Natal": "KwaZulu-Natal",
    "province_hhLimpopo": "Limpopo",
    "province_hhMpumalanga": "Mpumalanga",
    "province_hhNorth West": "North West",
    "province_hhNorthern Cape": "Northern Cape",
    "metro_hhNon-metro": "Non-metropolitan",
    "geotype_hh2": "Geographic type: code 2",
    "geotype_hh3": "Geographic type: code 3",
    "income_quintile_hh1": "Income Q1 (lowest)",
    "income_quintile_hh2": "Income Q2",
    "income_quintile_hh3": "Income Q3",
    "income_quintile_hh4": "Income Q4",
    "food_insecure_any_hh1": "Food insecurity",
    "household_size_hh": "Household size",
    "children_under18_hh": "Children under 18 present",
    "older_adults60_hh": "Adults aged 60+ present",
    "vehicle_owner_hh1": "Vehicle ownership",
    "any_medical_aid_hh1": "Any household medical aid",
}

term_order = list(term_labels.keys())

def format_or(row):
    return (
        f"{row['aOR']:.2f} "
        f"({row['CI_lower']:.2f}–{row['CI_upper']:.2f})"
    )

# ------------------------------------------------------------
# 3. Comparative M0–M2 table for each outcome
# ------------------------------------------------------------

for outcome in ["travel30_hh", "public_sector_hh"]:

    table = pd.DataFrame({
        "Term": term_order,
        "Predictor": [term_labels[t] for t in term_order]
    })

    for model_name, df in models.items():

        subset = df.loc[
            df["Outcome"] == outcome
        ].copy()

        assert not subset["Term"].duplicated().any()

        subset["Formatted"] = subset.apply(
            format_or, axis=1
        )

        table = table.merge(
            subset[["Term", "Formatted"]].rename(
                columns={"Formatted": model_name}
            ),
            on="Term",
            how="left",
            validate="one_to_one"
        )

    table = table.drop(columns="Term")

    output_path = (
        TABLES_DIR /
        f"table_regression_comparison_{outcome}.csv"
    )

    table.to_csv(output_path, index=False)

    print(f"\nREGRESSION TABLE: {outcome}")
    print("-" * 70)

    display(table.fillna("—"))

    print("Saved:", output_path)

# ------------------------------------------------------------
# 4. Consolidate AMEs
# ------------------------------------------------------------

ame_path = (
    TABLES_DIR / "m2_average_marginal_effects_ci.csv"
)

assert ame_path.exists()

ame = pd.read_csv(ame_path)

required_ame = {
    "Outcome", "Predictor", "AME_pp",
    "CI_lower_pp", "CI_upper_pp", "P_value"
}

assert required_ame.issubset(ame.columns)

ame_labels = {
    "metro_hh": "Non-metro vs Metro",
    "food_insecure_any_hh": "Food insecurity",
    "vehicle_owner_hh": "Vehicle ownership",
    "any_medical_aid_hh": "Any household medical aid",
}

ame["Predictor_label"] = ame["Predictor"].map(
    ame_labels
)

assert ame["Predictor_label"].notna().all()

ame["AME_95CI"] = ame.apply(
    lambda row: (
        f"{row['AME_pp']:+.2f} "
        f"({row['CI_lower_pp']:+.2f} to "
        f"{row['CI_upper_pp']:+.2f})"
    ),
    axis=1
)

ame_table = ame.pivot(
    index="Predictor_label",
    columns="Outcome",
    values="AME_95CI"
).reset_index()

ame_table = ame_table.rename(columns={
    "Predictor_label": "Predictor",
    "travel30_hh": "Travel ≥30 min (pp)",
    "public_sector_hh": "Public-sector facility (pp)"
})

ame_table["Order"] = ame_table["Predictor"].map({
    label: i for i, label in enumerate(
        ame_labels.values()
    )
})

ame_table = (
    ame_table.sort_values("Order")
    .drop(columns="Order")
    .reset_index(drop=True)
)

ame_output = (
    TABLES_DIR / "table_m2_average_marginal_effects.csv"
)

ame_table.to_csv(ame_output, index=False)

print("\nM2 AVERAGE MARGINAL EFFECTS TABLE")
print("-" * 70)

display(ame_table)

print("Saved:", ame_output)

print("\nCELL_31_TABLES_COMPLETED")



REGRESSION TABLE: travel30_hh
----------------------------------------------------------------------


,Predictor,M0,M1,M2
0,Intercept,0.04 (0.03–0.05),0.03 (0.02–0.04),0.04 (0.03–0.05)
1,Eastern Cape,2.69 (1.96–3.68),2.32 (1.71–3.16),2.27 (1.67–3.09)
2,Free State,2.74 (1.91–3.93),2.47 (1.73–3.54),2.42 (1.69–3.47)
3,Gauteng,1.39 (1.01–1.90),1.27 (0.93–1.73),1.25 (0.92–1.71)
4,KwaZulu-Natal,2.03 (1.47–2.79),1.82 (1.33–2.51),1.78 (1.30–2.45)
5,Limpopo,1.76 (1.26–2.46),1.90 (1.36–2.66),1.86 (1.33–2.61)
6,Mpumalanga,2.57 (1.83–3.62),2.46 (1.75–3.46),2.44 (1.74–3.44)
7,North West,3.03 (2.13–4.33),2.78 (1.96–3.95),2.76 (1.94–3.93)
8,Northern Cape,2.77 (1.89–4.06),2.33 (1.61–3.39),2.33 (1.60–3.38)
9,Non-metropolitan,1.65 (1.37–1.99),1.55 (1.29–1.86),1.55 (1.29–1.86)


Saved: C:\Users\tsholofelom\Documents\Research Projects\ghs-2025-healthcare-inequality\outputs\tables\table_regression_comparison_travel30_hh.csv

REGRESSION TABLE: public_sector_hh
----------------------------------------------------------------------


,Predictor,M0,M1,M2
0,Intercept,1.13 (0.99–1.29),0.22 (0.18–0.28),1.04 (0.82–1.32)
1,Eastern Cape,1.78 (1.48–2.14),1.09 (0.87–1.36),0.97 (0.76–1.25)
2,Free State,1.65 (1.33–2.07),1.01 (0.77–1.32),0.86 (0.62–1.19)
3,Gauteng,1.61 (1.39–1.87),1.42 (1.17–1.73),1.34 (1.09–1.66)
4,KwaZulu-Natal,2.02 (1.68–2.42),1.80 (1.44–2.24),1.93 (1.52–2.46)
5,Limpopo,1.18 (0.92–1.52),1.28 (0.98–1.69),1.27 (0.93–1.72)
6,Mpumalanga,1.19 (0.94–1.51),1.05 (0.81–1.38),1.16 (0.86–1.56)
7,North West,0.98 (0.77–1.24),0.63 (0.49–0.82),0.80 (0.59–1.08)
8,Northern Cape,1.28 (0.95–1.71),0.73 (0.54–0.99),0.81 (0.57–1.16)
9,Non-metropolitan,1.55 (1.37–1.76),1.38 (1.19–1.60),1.44 (1.23–1.69)


Saved: C:\Users\tsholofelom\Documents\Research Projects\ghs-2025-healthcare-inequality\outputs\tables\table_regression_comparison_public_sector_hh.csv

M2 AVERAGE MARGINAL EFFECTS TABLE
----------------------------------------------------------------------


Outcome,Predictor,Public-sector facility (pp),Travel ≥30 min (pp)
0,Non-metro vs Metro,+3.11 (+1.75 to +4.48),+4.66 (+2.80 to +6.52)
1,Food insecurity,+4.30 (+3.04 to +5.55),+5.36 (+3.87 to +6.84)
2,Vehicle ownership,-16.07 (-17.72 to -14.43),-3.52 (-4.99 to -2.05)
3,Any household medical aid,-35.67 (-37.97 to -33.37),-0.23 (-2.07 to +1.60)


Saved: C:\Users\tsholofelom\Documents\Research Projects\ghs-2025-healthcare-inequality\outputs\tables\table_m2_average_marginal_effects.csv

CELL_31_TABLES_COMPLETED


In [53]:

# ============================================================
# FINAL M2 MODEL DIAGNOSTICS
# ============================================================

import subprocess

r_script = r"""
suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu = "fail")

outcomes <- c("travel30_hh", "public_sector_hh")

diagnostics <- list()

for (outcome in outcomes) {

    d <- read.csv(
        paste0("data/regression/", outcome, "_regression.csv")
    )

    d$province_hh <- relevel(
        factor(d$province_hh), "Western Cape"
    )
    d$metro_hh <- relevel(
        factor(d$metro_hh), "Metro"
    )
    d$geotype_hh <- relevel(
        factor(d$geotype_hh), "1"
    )
    d$income_quintile_hh <- relevel(
        factor(d$income_quintile_hh), "5"
    )
    d$food_insecure_any_hh <- relevel(
        factor(d$food_insecure_any_hh), "0"
    )
    d$vehicle_owner_hh <- relevel(
        factor(d$vehicle_owner_hh), "0"
    )
    d$any_medical_aid_hh <- relevel(
        factor(d$any_medical_aid_hh), "0"
    )

    design <- svydesign(
        ids = ~psu,
        strata = ~stratum,
        weights = ~house_wgt,
        data = d,
        nest = TRUE
    )

    formula <- as.formula(paste(
        outcome,
        "~ province_hh + metro_hh + geotype_hh",
        "+ income_quintile_hh + food_insecure_any_hh",
        "+ household_size_hh + children_under18_hh",
        "+ older_adults60_hh",
        "+ vehicle_owner_hh + any_medical_aid_hh"
    ))

    fit <- svyglm(
        formula,
        design = design,
        family = quasibinomial()
    )

    X <- model.matrix(fit)
    probabilities <- as.numeric(fitted(fit))

    # Standardised matrix condition number:
    # excludes the intercept and rescales predictors.
    X_no_intercept <- X[
        , colnames(X) != "(Intercept)", drop = FALSE
    ]

    X_scaled <- scale(X_no_intercept)

    condition_number <- kappa(
        X_scaled,
        exact = TRUE
    )

    # Variance inflation factors for individual
    # dummy-coded columns. These are diagnostic
    # indicators, not generalized VIFs for factors.
    R <- cor(X_no_intercept)

    individual_vif <- diag(solve(R))

    geographic_cells <- aggregate(
        rep(1, nrow(d)),
        by = list(
            province = d$province_hh,
            metro = d$metro_hh,
            geotype = d$geotype_hh
        ),
        FUN = sum
    )

    names(geographic_cells)[4] <- "n"

    result <- data.frame(
        Outcome = outcome,
        N = nrow(d),
        PSU = length(unique(d$psu)),
        Strata = length(unique(d$stratum)),
        Design_df = degf(design),
        Converged = fit$converged,
        Model_rank = fit$rank,
        Coefficients = length(coef(fit)),
        Residual_df = fit$df.residual,
        Condition_number = condition_number,
        Maximum_dummy_VIF = max(individual_vif),
        Minimum_fitted_probability = min(probabilities),
        Maximum_fitted_probability = max(probabilities),
        Fitted_below_0_001 = sum(probabilities < 0.001),
        Fitted_above_0_999 = sum(probabilities > 0.999),
        Geographic_cells_below_30 = sum(
            geographic_cells$n < 30
        ),
        Minimum_geographic_cell_n = min(
            geographic_cells$n
        )
    )

    diagnostics[[outcome]] <- result

    cat("\nOUTCOME:", outcome, "\n")
    cat("----------------------------------------\n")
    cat("Households:", nrow(d), "\n")
    cat("PSUs:", result$PSU, "\n")
    cat("Strata:", result$Strata, "\n")
    cat("Design df:", result$Design_df, "\n")
    cat("Converged:", result$Converged, "\n")
    cat(
        "Model rank:",
        result$Model_rank, "/",
        result$Coefficients, "\n"
    )
    cat(
        "Residual df:",
        result$Residual_df, "\n"
    )
    cat(
        "Scaled condition number:",
        round(condition_number, 3), "\n"
    )
    cat(
        "Maximum dummy-column VIF:",
        round(max(individual_vif), 3), "\n"
    )
    cat(
        "Fitted probability range:",
        sprintf("%.6f", min(probabilities)),
        "to",
        sprintf("%.6f", max(probabilities)),
        "\n"
    )
    cat(
        "Fitted probabilities < 0.001:",
        sum(probabilities < 0.001), "\n"
    )
    cat(
        "Fitted probabilities > 0.999:",
        sum(probabilities > 0.999), "\n"
    )
    cat(
        "Geographic cells with n < 30:",
        sum(geographic_cells$n < 30), "\n"
    )
    cat(
        "Smallest geographic cell:",
        min(geographic_cells$n), "\n"
    )

    cat("\nHighest individual-column VIFs:\n")

    print(
        head(
            sort(individual_vif, decreasing = TRUE),
            6
        )
    )

    stopifnot(
        fit$converged,
        fit$rank == length(coef(fit)),
        all(is.finite(probabilities)),
        all(probabilities > 0),
        all(probabilities < 1),
        all(is.finite(individual_vif))
    )
}

diagnostics_table <- do.call(
    rbind, diagnostics
)

write.csv(
    diagnostics_table,
    "outputs/tables/m2_model_diagnostics.csv",
    row.names = FALSE
)

cat("\nM2_MODEL_DIAGNOSTICS_COMPLETED\n")
"""

script_path = (
    ROOT / "data" / "regression" /
    "m2_model_diagnostics.R"
)

script_path.write_text(r_script, encoding="utf-8")

result = subprocess.run(
    [str(RSCRIPT), "--vanilla", str(script_path)],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
    timeout=180
)

print("R RETURN CODE:", result.returncode)

print("\nR OUTPUT:")
print(result.stdout.strip() or "(empty)")

if result.stderr.strip():
    print("\nR MESSAGES:")
    print(result.stderr.strip())

if result.returncode == 0:
    assert "M2_MODEL_DIAGNOSTICS_COMPLETED" in result.stdout
    print("\nM2 diagnostics completed successfully.")
else:
    print("\nM2 diagnostics failed. Review the messages above.")


R RETURN CODE: 0

R OUTPUT:
OUTCOME: travel30_hh 
----------------------------------------
Households: 19481 
PSUs: 3212 
Strata: 248 
Design df: 2964 
Converged: TRUE 
Model rank: 22 / 22 
Residual df: 2943 
Scaled condition number: 5.498 
Maximum dummy-column VIF: 3.034 
Fitted probability range: 0.024036 to 0.618144 
Fitted probabilities < 0.001: 0 
Fitted probabilities > 0.999: 0 
Geographic cells with n < 30: 6 
Smallest geographic cell: 2 

Highest individual-column VIFs:
      province_hhGauteng      income_quintile_hh1      income_quintile_hh2 
                3.033532                 3.004972                 2.923391 
      province_hhLimpopo province_hhKwaZulu-Natal      income_quintile_hh3 
                2.657133                 2.584861                 2.455291 

OUTCOME: public_sector_hh 
----------------------------------------
Households: 19494 
PSUs: 3212 
Strata: 248 
Design df: 2964 
Converged: TRUE 
Model rank: 22 / 22 
Residual df: 2943 
Scaled condition number: 5

In [55]:

# ============================================================
# FINAL VERIFICATION AND OUTPUT MANIFEST
# ============================================================

import json
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. Verify all primary outputs
# ------------------------------------------------------------

required_outputs = {
    "M0 geographic regression":
        "m0_geographic_regression.csv",

    "M1 socioeconomic regression":
        "m1_socioeconomic_regression.csv",

    "M2 fully adjusted regression":
        "m2_fully_adjusted_regression.csv",

    "Nested model comparisons":
        "nested_model_summary.csv",

    "Standardised income probabilities":
        "m2_standardised_income_probabilities_point.csv",

    "AME point estimates":
        "m2_average_marginal_effects_point.csv",

    "AME confidence intervals":
        "m2_average_marginal_effects_ci.csv",

    "M2 diagnostics":
        "m2_model_diagnostics.csv",

    "Travel regression comparison":
        "table_regression_comparison_travel30_hh.csv",

    "Public-sector regression comparison":
        "table_regression_comparison_public_sector_hh.csv",

    "Publication AME table":
        "table_m2_average_marginal_effects.csv",
}

manifest_files = {}

for description, filename in required_outputs.items():

    path = TABLES_DIR / filename

    assert path.exists(), f"Missing output: {filename}"

    df = pd.read_csv(path)

    assert len(df) > 0, f"Empty output: {filename}"

    manifest_files[description] = {
        "path": str(path.relative_to(ROOT)),
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "size_bytes": int(path.stat().st_size)
    }

    print(
        f"PASS | {description} | "
        f"{len(df)} rows × {len(df.columns)} columns"
    )

# ------------------------------------------------------------
# 2. Verify M2 sample and model diagnostics
# ------------------------------------------------------------

diagnostics = pd.read_csv(
    TABLES_DIR / "m2_model_diagnostics.csv"
)

expected_samples = {
    "travel30_hh": 19481,
    "public_sector_hh": 19494
}

assert set(diagnostics["Outcome"]) == set(
    expected_samples
)

for outcome, expected_n in expected_samples.items():

    row = diagnostics.loc[
        diagnostics["Outcome"] == outcome
    ]

    assert len(row) == 1

    row = row.iloc[0]

    assert int(row["N"]) == expected_n
    assert int(row["PSU"]) == 3212
    assert int(row["Strata"]) == 248
    assert int(row["Design_df"]) == 2964
    assert int(row["Model_rank"]) == 22
    assert int(row["Coefficients"]) == 22

    assert str(row["Converged"]).lower() == "true"

print("\nPASS | M2 samples and survey design verified")

# ------------------------------------------------------------
# 3. Verify AME estimates against Cell 29
# ------------------------------------------------------------

ame_point = pd.read_csv(
    TABLES_DIR / "m2_average_marginal_effects_point.csv"
)

ame_ci = pd.read_csv(
    TABLES_DIR / "m2_average_marginal_effects_ci.csv"
)

ame_check = ame_point.merge(
    ame_ci,
    on=["Outcome", "Predictor"],
    suffixes=("_point", "_ci"),
    validate="one_to_one"
)

assert len(ame_check) == 8

assert np.allclose(
    ame_check["AME_point"],
    ame_check["AME_ci"],
    rtol=1e-8,
    atol=1e-10
)

assert (
    (ame_ci["CI_lower"] <= ame_ci["AME"]) &
    (ame_ci["AME"] <= ame_ci["CI_upper"])
).all()

assert (
    ame_ci["P_value"].between(0, 1)
).all()

print("PASS | All eight AMEs and confidence intervals verified")

# ------------------------------------------------------------
# 4. Verify regression comparison tables
# ------------------------------------------------------------

for outcome in expected_samples:

    table = pd.read_csv(
        TABLES_DIR /
        f"table_regression_comparison_{outcome}.csv"
    )

    assert len(table) == 22

    assert {"Predictor", "M0", "M1", "M2"}.issubset(
        table.columns
    )

    assert table["M2"].notna().all()

    print(f"PASS | M0–M2 regression table: {outcome}")

# ------------------------------------------------------------
# 5. Save completion manifest
# ------------------------------------------------------------

manifest = {
    "notebook": "03_adjusted_regression.ipynb",
    "status": "Primary regression analysis completed",
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "study": (
        "Who Is Being Left Behind? "
        "Socioeconomic and Geographic Inequalities "
        "in Healthcare Access in South Africa"
    ),
    "survey": "Statistics South Africa GHS 2025",
    "analysis_unit": "Household",
    "method": (
        "Survey-weighted logistic regression "
        "with PSU-, stratum-, and weight-adjusted inference"
    ),
    "model_sequence": ["M0", "M1", "M2"],
    "outcomes": {
        outcome: {
            "complete_case_n": n,
            "psu": 3212,
            "strata": 248,
            "design_df": 2964
        }
        for outcome, n in expected_samples.items()
    },
    "primary_outputs": manifest_files,
    "pending_sensitivity_checks": [
        "Verify official geographic-type code definitions",
        "Investigate sparse geographic combinations",
        "Investigate extreme public-sector fitted probabilities",
        "Assess complete-case exclusions",
        "Assess alternative geographic specifications",
        "Check model-standardisation support"
    ],
    "interpretation_notes": [
        "Cross-sectional associations, not causal effects",
        "Non-metropolitan does not necessarily mean rural",
        "Usual facility sector is not realised healthcare utilisation",
        "Travel time is not a direct measure of healthcare quality",
        "Primary models use household survey weights"
    ]
}

manifest_path = (
    TABLES_DIR / "notebook03_completion_manifest.json"
)

manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("\nManifest saved:", manifest_path)

print("\n" + "=" * 60)
print("NOTEBOOK 03 — FINAL VERIFICATION COMPLETED")
print("=" * 60)
print("Primary regression outputs verified.")
print("Ready to proceed to Notebook 04.")


PASS | M0 geographic regression | 24 rows × 7 columns
PASS | M1 socioeconomic regression | 40 rows × 8 columns
PASS | M2 fully adjusted regression | 44 rows × 8 columns
PASS | Nested model comparisons | 6 rows × 6 columns
PASS | Standardised income probabilities | 10 rows × 4 columns
PASS | AME point estimates | 8 rows × 5 columns
PASS | AME confidence intervals | 8 rows × 11 columns
PASS | M2 diagnostics | 2 rows × 17 columns
PASS | Travel regression comparison | 22 rows × 4 columns
PASS | Public-sector regression comparison | 22 rows × 4 columns
PASS | Publication AME table | 4 rows × 3 columns

PASS | M2 samples and survey design verified
PASS | All eight AMEs and confidence intervals verified
PASS | M0–M2 regression table: travel30_hh
PASS | M0–M2 regression table: public_sector_hh

Manifest saved: C:\Users\tsholofelom\Documents\Research Projects\ghs-2025-healthcare-inequality\outputs\tables\notebook03_completion_manifest.json

NOTEBOOK 03 — FINAL VERIFICATION COMPLETED
Primary reg